# Spark Interview Prep — Easy Level
## NYC Taxi & Limousine Commission (TLC) Fleet Registry

**Dataset:** NYC TLC Cab Registry (`Cabs.csv`) — ~8,970 rows, 14 columns

**Topics Covered:**
| # | Topic | Core Concept |
|---|-------|--------------|
| 1 | Filter Pushdown | Push filters to data source to reduce I/O |
| 2 | Column Pruning | Read only the columns you need |
| 3 | Cache vs Persist | Avoid recomputation across multiple actions |
| 4 | Repartition vs Coalesce | Control parallelism without unnecessary shuffles |
| 5 | CSV vs Parquet | Columnar formats, schema, compression |
| 6 | Partition Pruning | Skip entire directories at read time |

**How to use this notebook:**
1. Run the setup cell first
2. For each topic: read the business scenario, attempt the optimization exercise yourself
3. Compare your plan output against the Expected Plan Discussion sections
4. The solution cell is at the bottom of each topic — resist peeking!


In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.storagelevel import StorageLevel
import time

active = SparkSession.getActiveSession()
if active: active.stop()

spark = SparkSession.builder \
    .appName("SparkInterviewPrep-Easy") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
CABS_PATH = r"C:\Users\PS\Documents\Python-Exp\RawData\Cabs.csv"
SCRATCH    = r"C:\Users\PS\Documents\Python-Exp\RawData\scratch"
print("Spark version:", spark.version)
print("AQE enabled :", spark.conf.get("spark.sql.adaptive.enabled"))


---
## Topic 1 — Filter Pushdown

### Business Scenario
| Dimension | Detail |
|-----------|--------|
| **Team** | NYC TLC Fleet Operations |
| **Goal** | Extract only **ACTIVE** cabs for real-time dispatch routing |
| **Data Volume** | 8,970 rows CSV today; growing to 50M rows Parquet in production |
| **Pain Point** | Full table scan every query — I/O bound, slow dashboard |
| **SLA** | Dashboard must refresh in < 3 s |

### Dataset Description
```
Cabs.csv (~8,970 rows, 14 columns)
  CabNumber            string   -- Taxi medallion number e.g. T802127C
  VehicleLicenseNumber string   -- DMV license
  Name                 string   -- Owner/operator name
  LicenseType          string   -- OWNER MUST DRIVE | NAMED DRIVER | CORPORATE
  Active               string   -- YES | NO
  PermitLicenseNumber  string   -- nullable
  VehicleVinNumber     string
  WheelchairAccessible string   -- nullable
  VehicleYear          int
  VehicleType          string   -- nullable
  TelephoneNumber      string
  Website              string   -- nullable
  Address              string
  LastDateUpdated      string   -- MM/dd/yyyy
```

### Why Filter Pushdown Matters
Without pushdown: **Spark reads ALL bytes** from storage, deserializes every row into JVM objects, then applies the filter in memory.

With pushdown (Parquet + `PushedFilters`): **The storage layer (Parquet reader) skips row groups** whose statistics show they cannot satisfy the predicate. For a 50 M-row table this can reduce I/O by 60-90%.

```
CSV  (no pushdown possible):  Read 100% bytes → filter in Spark
Parquet (pushdown):           Read ~10% bytes → filter skipped at source
```


In [ ]:
# ── Topic 1: Sample Data Setup ──────────────────────────────────────────────
# Read the real Cabs.csv and cache it as our "source of truth"

cabs_schema = StructType([
    StructField("CabNumber",            StringType(), True),
    StructField("VehicleLicenseNumber", StringType(), True),
    StructField("Name",                 StringType(), True),
    StructField("LicenseType",          StringType(), True),
    StructField("Active",               StringType(), True),
    StructField("PermitLicenseNumber",  StringType(), True),
    StructField("VehicleVinNumber",     StringType(), True),
    StructField("WheelchairAccessible", StringType(), True),
    StructField("VehicleYear",          IntegerType(), True),
    StructField("VehicleType",          StringType(), True),
    StructField("TelephoneNumber",      StringType(), True),
    StructField("Website",              StringType(), True),
    StructField("Address",              StringType(), True),
    StructField("LastDateUpdated",      StringType(), True),
])

raw_cabs = spark.read.csv(CABS_PATH, header=True, schema=cabs_schema)
print(f"Total rows in Cabs.csv: {raw_cabs.count():,}")
print(f"Active=YES rows        : {raw_cabs.filter(F.col('Active')=='YES').count():,}")
print(f"Active=NO  rows        : {raw_cabs.filter(F.col('Active')=='NO').count():,}")
raw_cabs.printSchema()


### Problem Statement
The fleet operations team wrote a pipeline that:
1. Reads the full cab registry
2. Collects it to the driver
3. Filters active cabs in Python
4. Re-parallelizes the result

This is the worst possible pattern. Your job: rewrite it and prove the improvement using `explain()` and timing.


In [ ]:
# ── BAD CODE: Anti-Pattern — collect then filter in Python ───────────────────
# DO NOT DO THIS IN PRODUCTION

import time

t0 = time.time()

# PROBLEM 1: collect() pulls ALL rows to the driver — kills memory for large data
all_cabs = raw_cabs.collect()   # 8,970 rows → driver heap

# PROBLEM 2: filtering in Python — no Spark parallelism, no pushdown
active_cabs_python = [row for row in all_cabs if row["Active"] == "YES"]

# PROBLEM 3: re-parallelize — unnecessary round trip
active_df_bad = spark.createDataFrame(active_cabs_python, schema=cabs_schema)

count_bad = active_df_bad.count()
t1 = time.time()

print(f"BAD  approach — active cabs: {count_bad:,}  time: {t1-t0:.3f}s")
print()
print("=== Execution Plan (BAD) ===")
active_df_bad.explain(mode="formatted")


### Interview Questions — Topic 1: Filter Pushdown

Answer these before looking at the Expected Plan section below.

1. What does "predicate pushdown" mean in Spark, and at which layer of the execution stack does it occur?
2. Why can Spark push filters into Parquet files but **not** into plain CSV files?
3. You have a DataFrame with a UDF-based filter: `df.filter(my_udf(col('Active')) == 'YES')`. Will Catalyst push this filter down? Why or why not?
4. In the Spark UI, which tab and metric tells you how many rows were **eliminated by a pushed filter** before entering the Spark engine?
5. What is a Parquet **row group** and how do **column statistics** (min/max/null count) enable row-group skipping?
6. You call `df.select('A','B').filter(col('C') == 'YES')` — Spark throws an AnalysisException because `C` is not in the projection. How does Catalyst's rule `PushDownPredicates` interact with `ColumnPruning` to handle ordering of these operations?
7. A colleague claims: "I always put `.filter()` before `.select()` so Spark reads fewer columns." Is this correct? Explain what Catalyst actually does with the logical plan.
8. In a JDBC source, what Spark config or DataFrameReader option enables predicate pushdown, and what SQL is generated?


### Expected Logical Plan Discussion

When you call `df.filter(F.col('Active') == 'YES').explain(extended=True)` you will see:

```
== Analyzed Logical Plan ==
Filter (Active#7 = YES)
+- Relation [CabNumber#0, ...14 cols...] csv

== Optimized Logical Plan ==
Filter (isnotnull(Active#7) AND (Active#7 = YES))
+- Relation [CabNumber#0, ...14 cols...] csv
```

**Key observations:**
- Catalyst adds `isnotnull` automatically (null-safe predicate)
- The `Filter` node sits **above** the `Relation` node in the logical plan — for CSV this is unavoidable; the filter cannot descend into the source
- For Parquet the Optimized plan will show the filter **pushed into** the scan: `PushedFilters: [IsNotNull(Active), EqualTo(Active,YES)]`
- The `Relation` node represents the data source; its type (`csv` vs `parquet`) determines what optimizations are possible
- `ColumnPruning` rule fires separately to remove unreferenced columns from the `Relation`


### Expected Physical Plan Discussion

```
== Physical Plan ==
*(1) Filter (isnotnull(Active#7) AND (Active#7 = YES))
+- *(1) FileScan csv [...] Batched: false,
         DataFilters: [isnotnull(Active#7), (Active#7 = YES)],
         Format: CSV,
         PushedFilters: [],          <-- CSV: NOTHING pushed
         ReadSchema: struct<...>
```

For Parquet:
```
== Physical Plan ==
*(1) FileScan parquet [...]
     Batched: true,
     DataFilters: [],
     Format: Parquet,
     PushedFilters: [IsNotNull(Active), EqualTo(Active,YES)],  <-- pushed!
     ReadSchema: struct<CabNumber:string,...>
```

**Physical plan node glossary:**
| Node | Meaning |
|------|---------|
| `*(N)` prefix | Whole-stage code-generated stage N |
| `FileScan` | Read from storage |
| `PushedFilters` | Filters evaluated **inside the file reader** — zero Spark tasks |
| `DataFilters` | Filters evaluated **after** deserialization — costs CPU per row |
| `ReadSchema` | The columns actually decoded from storage |
| `Batched: true` | Vectorized (columnar) read — ~10x faster than row-by-row |


### Spark UI Investigation Guide — Filter Pushdown

After running a query, open `http://localhost:4040` and navigate:

**Jobs tab:**
- Count the number of jobs. A collect-then-filter approach generates extra jobs for the `createDataFrame` step.

**Stages tab → Stage Details:**
- `Input size / records` — for CSV this equals the full file size; for Parquet with pushdown it should be much smaller
- `Records Read` vs `Output Records` — large gap means filter is working

**SQL tab → Query Details → click the query:**
- Find the `FileScan` node in the DAG diagram
- Hover over it — you will see `PushedFilters`, `ReadSchema`, `number of files read`, `size of files read`
- A healthy Parquet scan shows `PushedFilters` populated and `size of files read` significantly less than total file size

**Tasks tab (within a Stage):**
- `Input Size` per task — uniform distribution means good partitioning
- `Duration` — outlier tasks reveal data skew
- `GC Time` — high GC (>5% of task duration) means the filter is too late and too much data was deserialized

**Metrics to record in interviews:**
```
CSV  scan: Input=45 MB, Records=8,970 → Filter → Output=7,234
Parquet  : Input=3.1 MB (skipped 93%), Records=7,234 → No filter node needed
```


In [ ]:
# ── OPTIMIZATION EXERCISE: Your turn ────────────────────────────────────────
# TODO 1: Read Cabs.csv with the predefined schema (avoid schema inference cost)
# TODO 2: Apply the filter BEFORE any collect() — keep data distributed
# TODO 3: Write the filtered result to Parquet (SCRATCH dir) — do this ONCE
# TODO 4: Read back from Parquet and apply the same filter
# TODO 5: Call .explain(mode='formatted') on both approaches
# TODO 6: Time both approaches and print the speedup ratio

import os
parquet_path = os.path.join(SCRATCH, "cabs_active_parquet")

# YOUR CODE HERE
# approach_1_df = ...
# approach_2_df = ...


In [ ]:
# ── PERFORMANCE BENCHMARKING ─────────────────────────────────────────────────
import os

parquet_path = os.path.join(SCRATCH, "t1_cabs_parquet")

# Write full dataset to Parquet once (simulating an ETL landing zone)
raw_cabs.write.mode("overwrite").parquet(parquet_path)
print(f"Parquet written to: {parquet_path}")

# --- Benchmark A: CSV with in-Spark filter (correct but no pushdown) ---
t0 = time.time()
csv_filtered = (
    spark.read.csv(CABS_PATH, header=True, schema=cabs_schema)
         .filter(F.col("Active") == "YES")
)
csv_count = csv_filtered.count()
t_csv = time.time() - t0

# --- Benchmark B: Parquet with predicate pushdown ---
t0 = time.time()
parquet_filtered = (
    spark.read.parquet(parquet_path)
         .filter(F.col("Active") == "YES")
)
parquet_count = parquet_filtered.count()
t_parquet = time.time() - t0

print(f"\n{'Method':<30} {'Count':>8} {'Time (s)':>10}")
print("-" * 52)
print(f"{'CSV  + Spark filter':<30} {csv_count:>8,} {t_csv:>10.3f}")
print(f"{'Parquet + pushdown':<30} {parquet_count:>8,} {t_parquet:>10.3f}")
if t_parquet > 0:
    print(f"\nSpeedup: {t_csv/t_parquet:.1f}x  (at 8K rows — grows to 10-100x at 50M rows)")

print("\n=== CSV Explain ===")
csv_filtered.explain(mode="formatted")
print("\n=== Parquet Explain ===")
parquet_filtered.explain(mode="formatted")


### Production Best Practices — Filter Pushdown

- **Always store landing-zone data as Parquet or Delta Lake**, never plain CSV, so that filter pushdown is possible from the first read.
- **Place filters as early as possible** in your logical plan even though Catalyst reorders them — it makes your code readable and documents intent.
- **Avoid UDFs on filter columns.** A UDF is a black box to Catalyst and will block pushdown. Use built-in `F.*` functions instead.
- **Avoid `df.collect()` followed by Python-level filtering.** This moves data to the driver, kills parallelism, and crashes on large datasets. Keep filters distributed.
- **For JDBC sources**, set `.option("pushDownPredicate", "true")` (default) and verify the generated SQL in the physical plan — ensure the `WHERE` clause appears in the SQL sent to the DB.
- **Enable Bloom filter pushdown for Parquet** (`spark.sql.parquet.filterPushdown.bloomFilter=true`) when filtering on high-cardinality string columns — reduces row-group reads further.
- **Monitor `PushedFilters` in the SQL tab** of the Spark UI on every production query. An empty `PushedFilters` on a Parquet source is a red flag worth investigating.


### Common Interview Follow-up Questions — Filter Pushdown

1. Delta Lake supports **data skipping** via transaction log statistics. How does this differ from Parquet row-group statistics, and when does Delta's skipping kick in before the Parquet-level pushdown?
2. You have a compound predicate: `filter((col('A') == 'X') | (col('B') > 5))`. Parquet cannot push OR predicates in all Spark versions — how would you restructure the query to maximize pushdown?
3. Explain the difference between `PushedFilters` and `PartitionFilters` in a physical plan. Which one skips more data?
4. A filter on a Parquet file is pushed down, but Spark UI still shows the full file size as `Input Size`. What could cause this?
5. How does `spark.sql.parquet.filterPushdown` interact with `spark.sql.optimizer.metadataOnly`? When would you disable the metadata-only optimization?


In [ ]:
# ============ SOLUTION (scroll down) ============

# ── SOLUTION: Filter Pushdown — Correct Patterns ────────────────────────────

import os
parquet_path = os.path.join(SCRATCH, "t1_cabs_parquet")

# ── Pattern A: CSV — keep filter in Spark (never collect first) ──────────────
# Catalyst will apply the filter as a DataFilter (post-deserialization),
# but at least it runs in parallel across all executors.

t0 = time.time()
active_from_csv = (
    spark.read
         .csv(CABS_PATH, header=True, schema=cabs_schema)  # explicit schema = no inference scan
         .filter(F.col("Active") == "YES")                  # Spark-side filter, parallel
         .select("CabNumber", "Name", "LicenseType", "VehicleYear", "TelephoneNumber")
)
count_a = active_from_csv.count()
t_a = time.time() - t0

# ── Pattern B: Parquet — true predicate pushdown ─────────────────────────────
# PushedFilters in the physical plan = filter evaluated inside the Parquet reader
# before rows are deserialized into Spark InternalRow objects.

t0 = time.time()
active_from_parquet = (
    spark.read
         .parquet(parquet_path)                              # vectorized columnar read
         .filter(F.col("Active") == "YES")                  # this will appear in PushedFilters
         .select("CabNumber", "Name", "LicenseType", "VehicleYear", "TelephoneNumber")
)
count_b = active_from_parquet.count()
t_b = time.time() - t0

# ── Pattern C: UDF blocks pushdown — demonstration ──────────────────────────
from pyspark.sql.functions import udf

@udf(returnType=BooleanType())
def is_active_udf(val):
    """This innocent-looking UDF completely kills filter pushdown."""
    return val == "YES"

# This filter will NOT appear in PushedFilters — Catalyst cannot inspect UDF internals
blocked_pushdown = (
    spark.read.parquet(parquet_path)
         .filter(is_active_udf(F.col("Active")))   # pushdown BLOCKED
         .select("CabNumber", "Active")
)

print("=== Pattern A: CSV + Spark filter ===")
active_from_csv.explain(mode="formatted")

print("\n=== Pattern B: Parquet + pushdown ===")
active_from_parquet.explain(mode="formatted")

print("\n=== Pattern C: UDF blocks pushdown ===")
blocked_pushdown.explain(mode="formatted")

print(f"\nCSV  time : {t_a:.3f}s  count={count_a:,}")
print(f"Parquet time: {t_b:.3f}s  count={count_b:,}")
print()
print("KEY INSIGHT: In the Parquet explain, 'PushedFilters' is populated.")
print("In the UDF explain, 'PushedFilters' is empty — full scan happens.")


---
## Topic 2 — Column Pruning

### Business Scenario
| Dimension | Detail |
|-----------|--------|
| **Team** | TLC Analytics — Dashboard Engineering |
| **Goal** | Display a cab summary table: medallion, owner, status, year |
| **Data Volume** | 8,970 rows × 14 columns today; 50 M rows × 80 columns in data warehouse |
| **Pain Point** | Query reads all 14 columns (incl. VIN, addresses, URLs) when only 4 are needed |
| **Impact** | 3.5× more I/O, 3.5× more deserialization CPU, 3.5× more memory pressure |

### Why Column Pruning Matters
Parquet is a **columnar format**: each column is stored in its own byte range within the file.
When Spark reads a Parquet file, it uses `ReadSchema` to tell the Parquet reader **which columns to decode**.
Columns not in `ReadSchema` are **never read from disk** — zero I/O, zero deserialization.

```
14-column Parquet file → select 4 columns → ReadSchema has 4 entries
  I/O cost  = 4/14 of the file  ≈ 28% of original
  CPU cost  = 4/14 deserialization ≈ 28% of original
```

For CSV this benefit does NOT exist — CSV is row-oriented; every byte must be scanned.

### Catalyst Rule: `ColumnPruning`
Catalyst's `ColumnPruning` optimizer rule propagates `select` projections **down** through the plan
to the `Relation` node. The physical planner then uses the pruned column list to build `ReadSchema`.


In [ ]:
# ── Topic 2: Data Setup ──────────────────────────────────────────────────────
# Ensure Parquet exists from Topic 1; re-write if needed
import os
parquet_path = os.path.join(SCRATCH, "t1_cabs_parquet")

if not os.path.exists(parquet_path):
    raw_cabs.write.mode("overwrite").parquet(parquet_path)
    print("Parquet written")
else:
    print("Parquet already exists")

# Print column sizes (approximate) by reading one column at a time
parquet_df = spark.read.parquet(parquet_path)
print(f"\nParquet schema ({len(parquet_df.columns)} columns):")
for c in parquet_df.columns:
    print(f"  {c}")


### Problem Statement
The analytics engineer read the full DataFrame and then selected 4 columns **after** doing an
expensive `groupBy` + `count`. Catalyst may or may not prune early enough.
Your job: rewrite to ensure the `ReadSchema` at the file-scan level contains **only the 4 needed columns**.


In [ ]:
# ── BAD CODE: Read all 14 columns, use 4 ────────────────────────────────────
import os
parquet_path = os.path.join(SCRATCH, "t1_cabs_parquet")

t0 = time.time()

# PROBLEM: No schema specified for CSV → triggers an extra scan for inference
# Then selects only 4 columns AFTER the full read is already planned
bad_df = spark.read.csv(CABS_PATH, header=True)  # schema inference = 1 extra CSV pass

# PROBLEM: .select() comes AFTER aggregation — Catalyst may not prune ReadSchema
bad_summary = (
    bad_df
    .groupBy("Active", "LicenseType")
    .agg(F.count("*").alias("cnt"),
         F.countDistinct("CabNumber").alias("unique_cabs"))
)
bad_summary.count()
t_bad = time.time() - t0

print(f"BAD approach time: {t_bad:.3f}s")
print("\n=== BAD: Full CSV read (note ReadSchema has ALL columns) ===")
spark.read.csv(CABS_PATH, header=True).explain(mode="formatted")


### Interview Questions — Topic 2: Column Pruning

1. Explain the Catalyst optimizer rule `ColumnPruning`. Which logical plan nodes does it target, and what does it emit?
2. A Parquet file has 80 columns. Your query selects 5. What appears in `ReadSchema` in the physical plan, and how does it affect I/O?
3. Why does column pruning **not** save I/O for CSV files even though Catalyst still rewrites the logical plan?
4. You join two DataFrames and then select a subset of columns. Does Catalyst push the projection **below** the join? What are the conditions for this rewrite?
5. What is the difference between `ReadSchema` and the full file schema? Where do you find each in the Spark UI?
6. A colleague uses `df.select('*')` then later `.drop('col1','col2')`. Does Catalyst prune these dropped columns from the file scan? Prove your answer with `explain()`.
7. How does Delta Lake's **column mapping** feature interact with Parquet column pruning?
8. You have a wide Parquet table (200 columns). Which Spark configuration controls the maximum number of fields Catalyst will try to prune before falling back to reading all?


### Expected Logical Plan Discussion

```
== Optimized Logical Plan (BAD — late select) ==
Aggregate [Active, LicenseType], [Active, LicenseType, count(1), count(distinct CabNumber)]
+- Relation [CabNumber, VehicleLicenseNumber, Name, ...(all 14 cols)...] csv

== Optimized Logical Plan (GOOD — early select + Parquet) ==
Aggregate [Active, LicenseType], [...]
+- Project [Active, LicenseType, CabNumber]     <-- Catalyst pushed projection down
   +- Relation [Active, LicenseType, CabNumber] parquet   <-- ReadSchema = 3 cols only
```

**What the Catalyst `ColumnPruning` rule does:**
1. Walks the logical plan bottom-up
2. At each operator, computes the set of columns that operator and all ancestors need
3. Injects a `Project` node below operators that reference fewer columns than the source
4. The physical planner translates the pruned `Project` into a narrowed `ReadSchema` on `FileScan`

**When pruning does NOT work:**
- CSV: `ReadSchema` is pruned in the plan, but the CSV parser still reads each full row as a string and splits on delimiter — no I/O savings
- `select('*')` followed by `drop()`: Catalyst resolves `*` to all columns first, then removes the dropped ones — the `ReadSchema` should still be pruned, but verify with `explain()`
- Star schemas with `SELECT *` in views: the view expansion happens before pruning


### Expected Physical Plan Discussion

```
GOOD — Parquet with early select:
== Physical Plan ==
*(2) HashAggregate(keys=[Active, LicenseType], functions=[count(1), count(CabNumber)])
+- Exchange hashpartitioning(Active, LicenseType, 8)
   +- *(1) HashAggregate(keys=[Active, LicenseType], functions=[partial_count(1), partial_count(CabNumber)])
      +- *(1) FileScan parquet
              ReadSchema: struct<CabNumber:string, Active:string, LicenseType:string>
              (only 3 of 14 columns!)
```

**Physical plan node glossary:**
| Node | Meaning |
|------|---------|
| `HashAggregate` | Two-phase aggregation: partial (map side) then final (reduce side) |
| `Exchange hashpartitioning` | Shuffle — sends rows to partitions by hash of group keys |
| `ReadSchema` | The columns decoded from Parquet — narrower = faster |
| `*(N)` | Whole-stage codegen boundary — fuses operators into one JVM method |


### Spark UI Investigation Guide — Column Pruning

**SQL tab → Query DAG:**
- Click the `FileScan parquet` node
- Check the tooltip/details for `ReadSchema` — count the number of fields
- If `ReadSchema` has more fields than your query needs, pruning failed

**Stages tab:**
- `Input Size` metric — compare wide vs narrow read for the same Parquet file
- A query selecting 4 of 14 columns should show `Input Size` ≈ 28% of the full-table read

**What to record for interviews:**
```
Full read  (14 cols): Input Size = 2.1 MB, Duration = 420 ms
Pruned read (4 cols): Input Size = 0.6 MB, Duration = 180 ms
Savings: 71% I/O reduction, 57% time reduction
```

**Common trap:** The `Input Size` shown in Stages is compressed bytes read from disk.
The actual deserialization savings are larger because wide rows have more CPU overhead.


In [ ]:
# ── OPTIMIZATION EXERCISE ────────────────────────────────────────────────────
import os
parquet_path = os.path.join(SCRATCH, "t1_cabs_parquet")

# TODO 1: Read from Parquet (not CSV) so column pruning actually saves I/O
# TODO 2: Select ONLY the columns needed for the aggregation BEFORE groupBy
#         Columns needed: CabNumber, Active, LicenseType, VehicleYear
# TODO 3: Call .explain(mode='formatted') and verify ReadSchema has <= 4 fields
# TODO 4: Time both bad (CSV, all cols) and good (Parquet, 4 cols) approaches
# TODO 5: Compare 'Input Size' in the explain output

NEEDED_COLS = ["CabNumber", "Active", "LicenseType", "VehicleYear"]

# YOUR CODE HERE


In [ ]:
# ── PERFORMANCE BENCHMARKING — Column Pruning ────────────────────────────────
import os
parquet_path = os.path.join(SCRATCH, "t1_cabs_parquet")

NEEDED_COLS = ["CabNumber", "Active", "LicenseType", "VehicleYear"]

# Benchmark: full CSV vs pruned Parquet
t0 = time.time()
full_csv_count = (
    spark.read.csv(CABS_PATH, header=True, schema=cabs_schema)
         .groupBy("Active", "LicenseType")
         .agg(F.count("*").alias("cnt"),
              F.countDistinct("CabNumber").alias("unique_cabs"),
              F.avg("VehicleYear").alias("avg_year"))
         .count()
)
t_full = time.time() - t0

t0 = time.time()
pruned_parquet_count = (
    spark.read.parquet(parquet_path)
         .select(NEEDED_COLS)                               # prune BEFORE groupBy
         .groupBy("Active", "LicenseType")
         .agg(F.count("*").alias("cnt"),
              F.countDistinct("CabNumber").alias("unique_cabs"),
              F.avg("VehicleYear").alias("avg_year"))
         .count()
)
t_pruned = time.time() - t0

print(f"{'Method':<40} {'Groups':>8} {'Time':>10}")
print("-" * 62)
print(f"{'CSV  all 14 cols':<40} {full_csv_count:>8} {t_full:>10.3f}s")
print(f"{'Parquet 4 cols (pruned)':<40} {pruned_parquet_count:>8} {t_pruned:>10.3f}s")

print("\n=== Parquet Pruned Plan ===")
(
    spark.read.parquet(parquet_path)
         .select(NEEDED_COLS)
         .groupBy("Active", "LicenseType")
         .agg(F.count("*"))
).explain(mode="formatted")


### Production Best Practices — Column Pruning

- **Always specify the schema explicitly** when reading CSV. Schema inference requires an extra full scan — doubles your CSV I/O before the query even starts.
- **Select only needed columns as the very first operation** after `.read`. This makes intent clear and gives Catalyst the best chance to prune the `ReadSchema`.
- **Prefer Parquet or Delta Lake** over CSV for any table read more than once — column pruning actually saves I/O only with columnar formats.
- **Audit `ReadSchema` on every production query** in the Spark UI SQL tab. A `ReadSchema` with more columns than your query's final output is a pruning leak.
- **Avoid `SELECT *`** in production code. Always name your columns — it prevents accidental column proliferation and makes schema evolution safer.
- **For streaming (Structured Streaming)**, column pruning works the same way — Kafka source pruning is limited, but Parquet-backed file stream sources benefit fully.
- **In Delta Lake**, use `OPTIMIZE` + `ZORDER BY` on high-selectivity columns so that column pruning combines with data skipping for maximum I/O reduction.


### Common Interview Follow-up Questions — Column Pruning

1. Explain what happens to column pruning when you use `df.withColumn('new_col', expr)` followed by a `select` that does not include the source column of `expr`. Does Catalyst eliminate the `withColumn` computation?
2. You have a struct column `address: struct<street:string, city:string, zip:string>`. You only need `address.city`. Does Parquet column pruning work at the **nested field** level?
3. A Parquet file written with schema version V1 is read by a job expecting schema V2 (one new column added). How does Parquet handle the missing column, and what appears in `ReadSchema`?
4. How does `spark.sql.optimizer.nestedSchemaPruning.enabled` change the behavior for nested structs and arrays?
5. You are reading a 200-column Parquet table but only need 3 columns. Your query still runs slowly. `ReadSchema` shows 3 columns. What else could cause slow performance?


In [ ]:
# ============ SOLUTION (scroll down) ============

# ── SOLUTION: Column Pruning — Correct Pattern ───────────────────────────────
import os
parquet_path = os.path.join(SCRATCH, "t1_cabs_parquet")

NEEDED_COLS = ["CabNumber", "Active", "LicenseType", "VehicleYear"]

# ── Step 1: Read Parquet and prune columns immediately ───────────────────────
# The .select() here causes Catalyst's ColumnPruning rule to narrow ReadSchema
# to exactly these 4 columns. The remaining 10 columns are never read from disk.
optimized_df = (
    spark.read
         .parquet(parquet_path)       # columnar format — pruning saves real I/O
         .select(NEEDED_COLS)         # prune BEFORE any transformation
)

# ── Step 2: Now do your transformations on the lean DataFrame ────────────────
summary = (
    optimized_df
    .filter(F.col("Active") == "YES")
    .groupBy("LicenseType", "VehicleYear")
    .agg(
        F.count("*").alias("total_cabs"),
        F.countDistinct("CabNumber").alias("unique_medallions")
    )
    .orderBy("VehicleYear", "LicenseType")
)

summary.show(20, truncate=False)

# ── Step 3: Verify ReadSchema in physical plan ───────────────────────────────
print("=== Optimized Physical Plan ===")
print("Look for 'ReadSchema' — it should contain only 4 fields:")
summary.explain(mode="formatted")

# ── Step 4: Demonstrate nested struct pruning (bonus) ───────────────────────
# Create a DataFrame with a struct column to show nested pruning
nested_schema = StructType([
    StructField("id",      IntegerType(), False),
    StructField("address", StructType([
        StructField("street", StringType(), True),
        StructField("city",   StringType(), True),
        StructField("zip",    StringType(), True),
    ]), True),
    StructField("year",    IntegerType(), True),
])

nested_path = os.path.join(SCRATCH, "nested_demo")
nested_data = [(i, (f"{i} Main St", "New York", f"100{i:02d}"), 2020 + (i % 5))
               for i in range(1, 101)]
nested_df = spark.createDataFrame(nested_data, schema=nested_schema)
nested_df.write.mode("overwrite").parquet(nested_path)

# Read and prune to nested field only
print("\n=== Nested struct pruning — only address.city ===")
spark.read.parquet(nested_path) \
     .select("id", "address.city") \
     .explain(mode="formatted")
# ReadSchema will show struct<city:string> not struct<street,city,zip>


---
## Topic 3 — Cache vs Persist

### Business Scenario
| Dimension | Detail |
|-----------|--------|
| **Team** | TLC Real-Time Dashboard |
| **Goal** | Compute 5 different KPIs from the same filtered active-cab dataset |
| **Data Volume** | 50 M row Parquet (simulated with `spark.range()`) |
| **Pain Point** | Each KPI action re-reads and re-filters the Parquet — 5× redundant I/O |
| **SLA** | Dashboard must complete all 5 KPIs in < 30 s |

### Storage Level Comparison
```
StorageLevel.MEMORY_ONLY          -- Default for cache(). Fast. Fails if too large → recomputes
StorageLevel.MEMORY_AND_DISK      -- Spills overflow to local disk. Slower but reliable.
StorageLevel.MEMORY_AND_DISK_SER  -- Serialized in memory. Less GC pressure. Slower deserialization.
StorageLevel.DISK_ONLY            -- No RAM usage. Slowest. Use for checkpointing large datasets.
StorageLevel.OFF_HEAP             -- Tungsten off-heap memory. No GC. Requires Unsafe memory mgmt.
```

### When to Cache vs When NOT to
```
CACHE when:
  - Same DataFrame is used in 2+ actions in the same Spark application
  - Recomputation is expensive (large shuffle, complex joins, multiple file reads)
  - DataFrame fits in executor memory (check df.count() vs spark.executor.memory)

DO NOT CACHE when:
  - DataFrame is used in only 1 action
  - Data is larger than total cluster memory
  - Data changes between uses (cache becomes stale)
  - Streaming context (use checkpointing instead)
```


In [ ]:
# ── Topic 3: Synthetic Large Dataset ────────────────────────────────────────
# Simulate a 500K-row fleet table using spark.range() + joins
# This makes caching impact measurable even on a local machine

import os
parquet_path = os.path.join(SCRATCH, "t1_cabs_parquet")

license_types = ["OWNER MUST DRIVE", "NAMED DRIVER", "CORPORATE"]
active_vals   = ["YES", "NO"]
vehicle_years = list(range(2010, 2024))

# Build a 500K-row synthetic fleet dataset
large_fleet = (
    spark.range(500_000)                               # id: 0..499999
         .withColumn("CabNumber",   F.concat(F.lit("T"), F.col("id").cast("string"), F.lit("C")))
         .withColumn("LicenseType", F.element_at(
             F.array([F.lit(x) for x in license_types]),
             (F.col("id") % 3 + 1).cast("int")
         ))
         .withColumn("Active",      F.when(F.col("id") % 5 == 0, "NO").otherwise("YES"))
         .withColumn("VehicleYear", F.lit(2010) + (F.col("id") % 14).cast("int"))
         .withColumn("Borough",     F.element_at(
             F.array([F.lit(b) for b in ["Manhattan","Brooklyn","Queens","Bronx","StatenIsland"]]),
             (F.col("id") % 5 + 1).cast("int")
         ))
         .withColumn("Trips30Days", (F.rand(42) * 300 + 50).cast("int"))
         .drop("id")
)

# Write to parquet so the 'read' cost is real (not just range() which is free)
fleet_path = os.path.join(SCRATCH, "t3_large_fleet")
large_fleet.write.mode("overwrite").parquet(fleet_path)
print(f"Written {large_fleet.count():,} rows to {fleet_path}")


### Problem Statement
The dashboard team has 5 KPI queries that all start from the same base:
"Active cabs with at least 100 trips in the last 30 days."

The bad implementation re-reads and re-filters the Parquet for each KPI.
Each action triggers a full scan + filter — 5 full scans instead of 1.

Your job: cache the base DataFrame and prove the speedup with timing.


In [ ]:
# ── BAD CODE: Recomputes the filtered DataFrame for every action ──────────────
import os
fleet_path = os.path.join(SCRATCH, "t3_large_fleet")

def get_active_fleet():  # called 5 times — 5 full Parquet scans!
    return (
        spark.read.parquet(fleet_path)
             .filter((F.col("Active") == "YES") & (F.col("Trips30Days") >= 100))
    )

t0 = time.time()

# KPI 1: Total active cabs
kpi1 = get_active_fleet().count()

# KPI 2: By borough
kpi2 = get_active_fleet().groupBy("Borough").count().collect()

# KPI 3: By license type
kpi3 = get_active_fleet().groupBy("LicenseType").count().collect()

# KPI 4: Average trips by year
kpi4 = get_active_fleet().groupBy("VehicleYear").agg(F.avg("Trips30Days")).collect()

# KPI 5: Top borough by total trips
kpi5 = get_active_fleet().groupBy("Borough").agg(F.sum("Trips30Days")).collect()

t_bad = time.time() - t0
print(f"BAD  (5 re-reads): {t_bad:.3f}s")
print(f"  KPI1 active cabs: {kpi1:,}")


### Interview Questions — Topic 3: Cache vs Persist

1. What is the difference between `df.cache()` and `df.persist(StorageLevel.MEMORY_AND_DISK)`?
2. When you call `df.cache()`, does Spark immediately read and store the data? If not, when does caching actually happen?
3. A cached DataFrame has 200 partitions. An executor fails and its 20 partitions are lost. What does Spark do? How does this differ between `MEMORY_ONLY` and `MEMORY_AND_DISK`?
4. You cache a DataFrame and notice the Spark UI shows 0 bytes cached after the `cache()` call. Why?
5. How do you force Spark to immediately populate the cache (materialize it) with a single call?
6. Explain the difference between `MEMORY_AND_DISK` and `MEMORY_AND_DISK_SER`. When would you prefer the serialized version?
7. You have two DataFrames: `df_a` (2 GB) and `df_b` (8 GB). Executor memory is 6 GB. You `cache()` `df_b` first, then `df_a`. What happens when Spark tries to cache `df_a`?
8. When should you use `df.checkpoint()` instead of `df.persist()`? What are the trade-offs?


### Expected Logical Plan Discussion

```
WITHOUT cache:
== Optimized Logical Plan (each KPI) ==
Aggregate [...]
+- Filter ((Active = YES) AND (Trips30Days >= 100))
   +- Relation [...] parquet          <-- full scan every time

WITH cache (after first action materializes the cache):
== Optimized Logical Plan (KPI 2-5) ==
Aggregate [...]
+- InMemoryRelation [...], StorageLevel(disk, memory, ..., 1 replicas)
      +- *(1) FileScan parquet [...]   <-- this is the cached plan
```

**Key observations:**
- The first action after `cache()` still shows `FileScan` — it reads from disk AND populates the cache
- Subsequent actions show `InMemoryRelation` — data served from executor memory
- `InMemoryRelation` appears in the Optimized Logical Plan, replacing the `FileScan + Filter` subtree
- The `InMemoryRelation` node stores the `StorageLevel` and the original plan (for recomputation on failure)


### Expected Physical Plan Discussion

```
WITH cache — after materialization:
== Physical Plan ==
*(2) HashAggregate(keys=[Borough], functions=[count(1)])
+- Exchange hashpartitioning(Borough, 8)
   +- *(1) HashAggregate(keys=[Borough], functions=[partial_count(1)])
      +- InMemoryTableScan [Borough, CabNumber, ...]
         +- InMemoryRelation [...]
```

**Physical plan node glossary:**
| Node | Meaning |
|------|---------|
| `InMemoryRelation` | The cached data descriptor — holds StorageLevel and original plan |
| `InMemoryTableScan` | Scans the cached RDD blocks from executor memory |
| `Exchange` | Shuffle boundary — data crosses executor boundaries |
| `HashAggregate` | Two-phase aggregate: partial (pre-shuffle) and final (post-shuffle) |

**In the Spark UI — Storage tab:**
- Each cached DataFrame appears as a named entry
- Shows: `Fraction Cached`, `Size in Memory`, `Size on Disk`
- A fraction of 1.0 means all partitions are cached
- Fraction < 1.0 means some partitions were evicted (use `MEMORY_AND_DISK` to prevent gaps)


### Spark UI Investigation Guide — Cache vs Persist

**Storage tab (http://localhost:4040/storage):**
- After calling `.count()` on a cached DF, refresh this tab
- See `RDD Name`, `Storage Level`, `Cached Partitions`, `Fraction Cached`, `Size in Memory`, `Size on Disk`
- A `Fraction Cached` < 1.0 means memory pressure — switch to `MEMORY_AND_DISK`

**SQL tab → individual query:**
- Uncached queries: DAG starts with `FileScan`
- Cached queries (after materialization): DAG starts with `InMemoryTableScan`
- The `InMemoryTableScan` node shows `numOutputRows` — should match cached row count

**Jobs tab:**
- Without cache: each KPI generates a job that includes a `FileScan` stage
- With cache: KPI 1 job includes `FileScan` (materializes cache); KPI 2-5 jobs show only `InMemoryTableScan` stage — no file I/O stage

**Key metrics to compare:**
```
Without cache: 5 jobs × (Scan stage + Agg stage) = 10 stages, ~5× file I/O
With    cache: 1 job with Scan+materialize + 4 jobs with InMemory only = 6 stages, 1× file I/O
```


In [ ]:
# ── OPTIMIZATION EXERCISE ────────────────────────────────────────────────────
import os
fleet_path = os.path.join(SCRATCH, "t3_large_fleet")

# TODO 1: Read the large fleet Parquet
# TODO 2: Filter to active cabs with Trips30Days >= 100
# TODO 3: Cache the filtered DataFrame
# TODO 4: Force materialization with a .count() action
# TODO 5: Run all 5 KPI queries and time the total
# TODO 6: Call .unpersist() after you are done
# TODO 7: Compare timing vs the bad approach above
# TODO 8: Show the explain() for one KPI — verify InMemoryRelation appears

# YOUR CODE HERE


In [ ]:
# ── PERFORMANCE BENCHMARKING — Cache vs Persist ──────────────────────────────
import os
fleet_path = os.path.join(SCRATCH, "t3_large_fleet")

def run_5_kpis(base_df, label):
    t0 = time.time()
    results = {}
    results["total"]        = base_df.count()
    results["by_borough"]   = base_df.groupBy("Borough").count().count()
    results["by_license"]   = base_df.groupBy("LicenseType").count().count()
    results["avg_trips"]    = base_df.agg(F.avg("Trips30Days")).collect()[0][0]
    results["top_borough"]  = base_df.groupBy("Borough").agg(F.sum("Trips30Days")) \
                                     .orderBy(F.desc("sum(Trips30Days)")).first()[0]
    elapsed = time.time() - t0
    print(f"[{label}] Completed 5 KPIs in {elapsed:.3f}s")
    print(f"  total active cabs: {results['total']:,}")
    print(f"  top borough by trips: {results['top_borough']}")
    return elapsed

base_filter = (
    spark.read.parquet(fleet_path)
         .filter((F.col("Active") == "YES") & (F.col("Trips30Days") >= 100))
)

# --- NO CACHE: each action triggers full Parquet scan + filter ---
t_nocache = run_5_kpis(base_filter, "NO CACHE")

# --- MEMORY_ONLY (default cache) ---
cached_mem = base_filter.cache()
cached_mem.count()  # materialize
t_mem = run_5_kpis(cached_mem, "MEMORY_ONLY")
cached_mem.unpersist()

# --- MEMORY_AND_DISK ---
cached_disk = base_filter.persist(StorageLevel.MEMORY_AND_DISK)
cached_disk.count()  # materialize
t_disk = run_5_kpis(cached_disk, "MEMORY_AND_DISK")
cached_disk.unpersist()

print(f"\n{'Method':<25} {'Time (5 KPIs)':>15} {'Speedup':>10}")
print("-" * 55)
print(f"{'No cache':<25} {t_nocache:>15.3f}s {'1.0x':>10}")
print(f"{'MEMORY_ONLY':<25} {t_mem:>15.3f}s {t_nocache/t_mem:>9.1f}x")
print(f"{'MEMORY_AND_DISK':<25} {t_disk:>15.3f}s {t_nocache/t_disk:>9.1f}x")


### Production Best Practices — Cache vs Persist

- **Always call `.count()` (or another action) immediately after `.cache()`** to materialize the cache and get an accurate timing baseline. `cache()` is lazy — the data is not stored until an action is triggered.
- **Always call `.unpersist()` when done.** Cached DataFrames hold executor memory until explicitly released or the `SparkContext` stops. Leaking caches causes OOM in long-running jobs.
- **Use `MEMORY_AND_DISK` as your production default**, not `MEMORY_ONLY`. In production, memory is shared with other jobs; `MEMORY_ONLY` silently falls back to recomputation if evicted. `MEMORY_AND_DISK` spills to local disk and avoids recomputing from source.
- **Name your cached DataFrames** with `df.createOrReplaceTempView('name')` or use `StorageLevel` with a descriptive tag so they are identifiable in the Spark UI Storage tab.
- **Check `Fraction Cached` in the UI.** If it is < 1.0, you have memory pressure. Either increase `spark.executor.memory`, switch to `MEMORY_AND_DISK_SER` (serialized = smaller footprint), or reduce the cached dataset size with earlier filtering.
- **Avoid caching DataFrames that are used only once.** The overhead of caching (serialize, store, GC pressure) exceeds the benefit if there is no reuse.
- **In iterative ML workloads** (MLlib, Spark ML), always cache the training dataset before fitting — the algorithm reads the data multiple times per iteration.


### Common Interview Follow-up Questions — Cache vs Persist

1. A cached DataFrame shows `Fraction Cached = 0.7` in the Spark UI. What does this mean operationally, and what happens when Spark tries to read a partition that was evicted?
2. You call `df.cache()` inside a function that is called in a loop. What happens to memory over time, and how would you fix it?
3. Explain the difference between `persist()` and `checkpoint()`. When would you use `checkpoint()` and what does it do to the lineage graph?
4. In a Spark Structured Streaming job, why can you not use `cache()` and what is the recommended alternative for stateful operations?
5. How does Tungsten's off-heap storage (`OFF_HEAP` StorageLevel) differ from on-heap storage in terms of GC behavior, and when is it beneficial?


In [ ]:
# ============ SOLUTION (scroll down) ============

# ── SOLUTION: Cache vs Persist — Correct Pattern ─────────────────────────────
import os
fleet_path = os.path.join(SCRATCH, "t3_large_fleet")

# ── Step 1: Build the expensive base DataFrame (read + multi-condition filter) ─
# This is what we want to avoid recomputing 5 times
active_fleet = (
    spark.read
         .parquet(fleet_path)
         .filter(
             (F.col("Active") == "YES") &
             (F.col("Trips30Days") >= 100)
         )
         .select("CabNumber", "LicenseType", "Borough", "VehicleYear", "Trips30Days")
         # select before cache: cache only the pruned columns, not all 7
)

# ── Step 2: Persist with MEMORY_AND_DISK (production-safe) ──────────────────
# MEMORY_AND_DISK: memory first, spills to executor local disk if evicted
# This prevents silent recomputation on executor failure or memory pressure
active_fleet = active_fleet.persist(StorageLevel.MEMORY_AND_DISK)

# ── Step 3: Materialize — MUST call an action to populate the cache ──────────
t0 = time.time()
n_cached = active_fleet.count()   # first action: reads Parquet AND stores in cache
t_materialize = time.time() - t0
print(f"Cache materialized: {n_cached:,} rows in {t_materialize:.3f}s")

# Verify in explain: InMemoryRelation should appear for all subsequent queries
print("\n=== Cached DataFrame Plan ===")
active_fleet.groupBy("Borough").count().explain(mode="formatted")

# ── Step 4: Run all 5 KPIs — each hits InMemoryTableScan, not FileScan ──────
t0 = time.time()

kpi1_total        = active_fleet.count()
kpi2_by_borough   = active_fleet.groupBy("Borough").agg(
                        F.count("*").alias("cabs"),
                        F.sum("Trips30Days").alias("total_trips")
                    ).orderBy(F.desc("total_trips"))
kpi3_by_license   = active_fleet.groupBy("LicenseType").count()
kpi4_avg_by_year  = active_fleet.groupBy("VehicleYear") \
                                 .agg(F.avg("Trips30Days").alias("avg_trips")) \
                                 .orderBy("VehicleYear")
kpi5_yearly_trend = active_fleet.groupBy("VehicleYear", "Borough") \
                                 .agg(F.count("*").alias("cabs")) \
                                 .orderBy("VehicleYear", F.desc("cabs"))

# Collect all KPIs
r2 = kpi2_by_borough.collect()
r3 = kpi3_by_license.collect()
r4 = kpi4_avg_by_year.collect()
r5 = kpi5_yearly_trend.collect()

t_kpis = time.time() - t0

print(f"\n5 KPIs (from cache) completed in {t_kpis:.3f}s")
print(f"Total active cabs (KPI1): {kpi1_total:,}")
print("\nKPI2 — Top boroughs:")
for row in r2[:3]: print(f"  {row['Borough']}: {row['cabs']:,} cabs, {row['total_trips']:,} trips")

# ── Step 5: Always unpersist when done ──────────────────────────────────────
active_fleet.unpersist()
print("\nCache released — memory returned to Spark.")


---
## Topic 4 — Repartition vs Coalesce

### Business Scenario
| Dimension | Detail |
|-----------|--------|
| **Team** | TLC Data Engineering — ETL Pipeline |
| **Goal** | Write daily active-cab snapshot to Parquet efficiently |
| **Data Volume** | 8,970 input rows → ~7,200 active rows → target: 4 output Parquet files |
| **Pain Point** | Using `repartition(4)` after filtering causes a full shuffle; `coalesce(4)` is cheaper |
| **Secondary Pain** | Over-partitioning (e.g., 200 partitions for 7K rows) creates small-files problem |

### Repartition vs Coalesce Decision Matrix
```
Operation         Shuffle?  Use When
─────────────────────────────────────────────────────────────────────────
repartition(n)    YES       Increasing partitions OR balancing skewed data
                            n > current partitions → MUST use repartition
                            Heavy skew in current partitions → repartition fixes it
                            Partition by column → repartition(n, 'col')

coalesce(n)       NO*       Decreasing partitions only (n < current)
                            Post-filter: 8970 rows → 7200 → coalesce(4)
                            *Narrow dependency: merges existing partitions
                            WARNING: can produce skewed partitions
```

### Small Files Problem
```
7,200 rows across 200 partitions = 36 rows/file = ~3 KB per Parquet file
Ideal Parquet file size: 128 MB – 1 GB
Impact of 3 KB files: 200× NameNode metadata overhead, slow listing, slow reads
Fix: coalesce(4) → 1,800 rows/file (still small for local data, but 50× better)
```

In [ ]:
# ── Topic 4: Setup ────────────────────────────────────────────────────────────
import os
parquet_path = os.path.join(SCRATCH, "t1_cabs_parquet")

if not os.path.exists(parquet_path):
    raw_cabs.write.mode("overwrite").parquet(parquet_path)

active_cabs = (
    spark.read.parquet(parquet_path)
         .filter(F.col("Active") == "YES")
         .select("CabNumber", "Name", "LicenseType", "Active", "VehicleYear", "TelephoneNumber")
)

print(f"Active cabs            : {active_cabs.count():,}")
print(f"Partitions after read  : {spark.read.parquet(parquet_path).rdd.getNumPartitions()}")
print(f"Partitions after filter: {active_cabs.rdd.getNumPartitions()}")
print("NOTE: filter does NOT change partition count — it only removes rows within each partition")

### Problem Statement
The ETL engineer used `repartition(4)` after filtering, causing an unnecessary full shuffle.
A junior engineer set `repartition(200)` thinking more partitions = more parallelism — creating 200 tiny 3 KB Parquet files.

Your job: replace `repartition` with `coalesce` where valid, prove no shuffle occurs via `explain()`, and fix the small-files problem.

In [ ]:
# ── BAD CODE: repartition() when coalesce() is sufficient ────────────────────
import os, glob
parquet_path = os.path.join(SCRATCH, "t1_cabs_parquet")
bad_out4     = os.path.join(SCRATCH, "t4_bad_reprt4")
bad_out200   = os.path.join(SCRATCH, "t4_bad_reprt200")

active = (
    spark.read.parquet(parquet_path)
         .filter(F.col("Active") == "YES")
         .select("CabNumber", "Name", "LicenseType", "Active", "VehicleYear")
)

# PROBLEM 1: repartition(4) when reducing → full shuffle Exchange for no reason
t0 = time.time()
active.repartition(4).write.mode("overwrite").parquet(bad_out4)
t_bad4 = time.time() - t0

# PROBLEM 2: repartition(200) → 200 tiny files (small-files problem)
t0 = time.time()
active.repartition(200).write.mode("overwrite").parquet(bad_out200)
t_bad200 = time.time() - t0

files4   = glob.glob(bad_out4   + "/part-*.parquet")
files200 = glob.glob(bad_out200 + "/part-*.parquet")
sz4   = [os.path.getsize(f) for f in files4]   if files4   else [0]
sz200 = [os.path.getsize(f) for f in files200] if files200 else [0]

print(f"repartition(4)  : {len(files4):>4} files, avg {sum(sz4)/len(sz4)/1024:>7.1f} KB, {t_bad4:.3f}s")
print(f"repartition(200): {len(files200):>4} files, avg {sum(sz200)/len(sz200)/1024:>7.1f} KB, {t_bad200:.3f}s")

print("\n=== repartition(4) plan — spot the Exchange ===")
active.repartition(4).explain(mode="formatted")

### Interview Questions — Topic 4: Repartition vs Coalesce

1. What is the fundamental algorithmic difference between `repartition(n)` and `coalesce(n)`? Why does `repartition` always trigger a shuffle while `coalesce` does not?
2. You have a DataFrame with 200 partitions and 7,200 rows. You call `coalesce(4)`. Describe the physical execution — which partitions are merged, and can this cause skew?
3. In which scenarios is it correct to use `repartition()` even when you are reducing the partition count?
4. What does the `Exchange` node in the physical plan represent? How do you distinguish a `RoundRobinPartitioning` Exchange (from `repartition`) from a `HashPartitioning` Exchange (from a shuffle join)?
5. You write a Parquet table with 200 partitions. Later reads are slow. How do you diagnose the small-files problem using Spark UI and file system metadata?
6. `repartition(n, col)` vs `repartition(n)` — what is the difference in the physical plan and when would you prefer the column-based form?
7. What is the rule of thumb for the ideal number of output Parquet files, and how do you calculate it given a target file size of 256 MB?
8. How does `spark.sql.files.maxPartitionBytes` interact with the number of partitions created during a Parquet read?

### Expected Logical Plan Discussion

```
repartition(4) — Logical Plan:
Repartition 4, true               <-- 'true' = shuffle=YES (full Exchange)
+- Filter (Active = YES)
   +- Relation [...] parquet

coalesce(4) — Logical Plan:
Repartition 4, false              <-- 'false' = shuffle=NO (narrow dependency)
+- Filter (Active = YES)
   +- Relation [...] parquet
```

**The boolean flag in `Repartition` is the key:**
- `true` → physical plan emits `Exchange RoundRobinPartitioning(n)` — full shuffle
- `false` → physical plan emits `Coalesce n` — merges adjacent partitions, no network I/O

**Why Catalyst cannot optimize away an explicit repartition:**
The user specified a partition count; Catalyst treats it as a hard requirement.
However, AQE can merge post-shuffle partitions via `CoalesceShufflePartitions`
if the downstream operator is also a shuffle.

### Expected Physical Plan Discussion

```
repartition(4):
== Physical Plan ==
Exchange RoundRobinPartitioning(4)        <-- full shuffle, even distribution
+- *(1) Filter (Active = YES)
   +- *(1) FileScan parquet [...]

coalesce(4):
== Physical Plan ==
Coalesce 4                               <-- NO Exchange node!
+- *(1) Filter (Active = YES)
   +- *(1) FileScan parquet [...]
```

**Physical plan node glossary:**
| Node | Meaning |
|------|---------|
| `Exchange RoundRobinPartitioning(n)` | Full shuffle — rows distributed evenly to n partitions |
| `Exchange HashPartitioning(cols, n)` | Full shuffle — rows hashed by column(s) to n partitions |
| `Coalesce n` | Narrow dependency — merge adjacent partitions, zero network I/O |
| `*(N)` prefix | Whole-stage codegen stage boundary |

**Shuffle cost breakdown:**
```
repartition(4): serialize all rows → write to shuffle spill files (disk) → network transfer → deserialize
coalesce(4):    merge partition metadata only — no serialization, no network, near-zero cost
```

### Spark UI Investigation Guide — Repartition vs Coalesce

**Stages tab:**
- `repartition(4)` query: **2 stages** — Stage 1 reads+filters, Stage 2 is the shuffle write
- `coalesce(4)` query: **1 stage** — no shuffle stage at all
- Presence of a second stage is the definitive indicator of a shuffle

**SQL tab → Query DAG:**
- `repartition(4)`: DAG shows `Exchange` node between `FileScan` and write operator
- `coalesce(4)`: DAG shows `Coalesce` node — no `Exchange`

**Stages tab → Shuffle Read/Write metrics:**
- `repartition(4)`: `Shuffle Write` in Stage 1 shows bytes serialized; `Shuffle Read` in Stage 2 shows bytes deserialized
- `coalesce(4)`: both `Shuffle Write` and `Shuffle Read` = 0 bytes

**File system check after write:**
```python
import glob, os
files = glob.glob(output_path + '/part-*.parquet')
sizes = [os.path.getsize(f) for f in files]
print(f'{len(files)} files, avg {sum(sizes)/len(sizes)/1024/1024:.1f} MB each')
# Healthy:  4-16 files, 128MB-1GB each
# Problem:  200 files, < 1 MB each → small-files problem
```

In [ ]:
# ── OPTIMIZATION EXERCISE ────────────────────────────────────────────────────
import os
parquet_path = os.path.join(SCRATCH, "t1_cabs_parquet")

active = (
    spark.read.parquet(parquet_path)
         .filter(F.col("Active") == "YES")
         .select("CabNumber", "Name", "LicenseType", "Active", "VehicleYear")
)

print(f"Partitions before: {active.rdd.getNumPartitions()}")

# TODO 1: Replace repartition(4) with coalesce(4)
# TODO 2: Call .explain(mode='formatted') — verify NO Exchange node appears
# TODO 3: Print getNumPartitions() before and after coalesce
# TODO 4: Time both approaches writing to SCRATCH and compare
# TODO 5: Count output files for each approach
# TODO 6: Demonstrate when repartition IS correct (increase partition count on large dataset)

# YOUR CODE HERE

In [ ]:
# ── PERFORMANCE BENCHMARKING — Repartition vs Coalesce ───────────────────────
import os, glob
parquet_path = os.path.join(SCRATCH, "t1_cabs_parquet")

active = (
    spark.read.parquet(parquet_path)
         .filter(F.col("Active") == "YES")
         .select("CabNumber", "Name", "LicenseType", "Active", "VehicleYear")
)

configs = [
    ("repartition(4)",   active.repartition(4)),
    ("coalesce(4)",      active.coalesce(4)),
    ("repartition(200)", active.repartition(200)),
    ("coalesce(1)",      active.coalesce(1)),
]

results = []
for label, df in configs:
    tag = label.replace("(","").replace(")","").replace(",","_").replace(" ","")
    out = os.path.join(SCRATCH, f"t4_bench_{tag}")
    t0 = time.time()
    df.write.mode("overwrite").parquet(out)
    elapsed = time.time() - t0
    files = glob.glob(out + "/part-*.parquet")
    sizes = [os.path.getsize(f) for f in files]
    avg_kb = (sum(sizes) / len(sizes) / 1024) if sizes else 0
    results.append((label, df.rdd.getNumPartitions(), elapsed, len(files), avg_kb))

print(f"{'Method':<22} {'Partitions':>12} {'Write(s)':>10} {'#Files':>8} {'AvgKB':>10}")
print("-" * 68)
for m, p, t, nf, kb in results:
    print(f"{m:<22} {p:>12} {t:>10.3f} {nf:>8} {kb:>10.1f}")

print("\n=== repartition(4) plan — Exchange present ===")
active.repartition(4).explain(mode="formatted")
print("\n=== coalesce(4) plan — no Exchange ===")
active.coalesce(4).explain(mode="formatted")

### Production Best Practices — Repartition vs Coalesce

- **Default to `coalesce(n)` when reducing partition count** after a filter. It avoids a full shuffle with identical output. Only switch to `repartition(n)` to increase partitions or fix heavy data skew.
- **Target 128 MB – 1 GB per output Parquet file.** Formula: `n_files = ceil(expected_compressed_bytes / 268_435_456)`. For a 50 GB compressed table targeting 256 MB files: `n = ceil(50*1024/256) = 200`.
- **Avoid `coalesce(1)` in production** except for tiny lookup tables. A single-partition write serializes everything through one executor — write bottleneck and OOM risk on large data.
- **Use `repartition(n, col)` to pre-partition for downstream joins.** If a join on `LicenseType` follows, `repartition(8, 'LicenseType')` routes matching keys to the same partition, letting Spark skip the join's own shuffle Exchange.
- **Monitor shuffle bytes** in the Spark UI Stages tab. An unnecessary `repartition` that shuffles gigabytes is a critical bottleneck — the first thing to eliminate.
- **Watch for the small-files problem** in incremental pipelines. If each micro-batch appends 200 files every hour, after 24 hours you have 4,800 tiny files. Use Delta Lake `OPTIMIZE` or periodic compaction jobs.
- **AQE `CoalesceShufflePartitions`** (`spark.sql.adaptive.coalescePartitions.enabled=true`) automatically reduces post-shuffle partition count at runtime. Reduces the need for manual `coalesce()` calls after joins and aggregations.

### Common Interview Follow-up Questions — Repartition vs Coalesce

1. AQE has `CoalesceShufflePartitions`. How does it interact with an explicit `coalesce(n)` call in your code? Which takes precedence and why?
2. You have a 10 TB Parquet table written with `repartition(100)`. Each file is 100 GB. Reads are slow. What is the correct fix — configure HDFS block splits, or change Spark read partitions?
3. Explain the lineage (RDD DAG) impact of `coalesce(1)`. Why can this cause OOM on the single executor handling the merged partition?
4. You call `df.repartition(8, 'Active')`. `Active` has only 2 distinct values (`YES`/`NO`). What happens to the other 6 partitions? How does this affect downstream parallelism?
5. In Delta Lake, how does `OPTIMIZE` (file compaction) differ from calling `coalesce()` in a Spark job, and which is preferred for production managed tables?

In [ ]:
# ============ SOLUTION (scroll down) ============

# ── SOLUTION: Repartition vs Coalesce ────────────────────────────────────────
import os, glob
parquet_path = os.path.join(SCRATCH, "t1_cabs_parquet")
good_output  = os.path.join(SCRATCH, "t4_good_output")

active = (
    spark.read.parquet(parquet_path)
         .filter(F.col("Active") == "YES")
         .select("CabNumber", "Name", "LicenseType", "Active", "VehicleYear")
)

# ── Step 1: Partition count audit ───────────────────────────────────────────
n_raw    = spark.read.parquet(parquet_path).rdd.getNumPartitions()
n_filter = active.rdd.getNumPartitions()
print(f"Partitions after read  : {n_raw}")
print(f"Partitions after filter: {n_filter}  (unchanged — filter is a narrow transformation)")

# ── Step 2: Use coalesce() — reducing, so no shuffle needed ─────────────────
# coalesce merges n_filter partitions into 4 by combining adjacent tasks.
# No data crosses executor boundaries. No serialization. No network.
coalesced = active.coalesce(4)
print(f"Partitions after coalesce(4): {coalesced.rdd.getNumPartitions()}")

# ── Step 3: Verify no Exchange in plan ──────────────────────────────────────
print("\n=== coalesce(4) plan — Coalesce node, no Exchange ===")
coalesced.explain(mode="formatted")

# ── Step 4: Write and confirm file count ─────────────────────────────────────
t0 = time.time()
coalesced.write.mode("overwrite").parquet(good_output)
t_write = time.time() - t0

files = glob.glob(good_output + "/part-*.parquet")
sizes = [os.path.getsize(f) for f in files]
print(f"\nWrite time   : {t_write:.3f}s")
print(f"Output files : {len(files)}")
print(f"Avg file size: {sum(sizes)/max(len(sizes),1)/1024:.1f} KB")

# ── Step 5: When repartition IS correct — increasing parallelism ─────────────
# The 500K-row large fleet was written with limited partitions.
# Before a wide aggregation, increase parallelism — coalesce CANNOT do this.
large = spark.read.parquet(os.path.join(SCRATCH, "t3_large_fleet"))
n_large = large.rdd.getNumPartitions()
print(f"\nLarge fleet partitions: {n_large}")

if n_large < 16:
    # Must use repartition — we are INCREASING the partition count
    upscaled = large.repartition(16)
    print(f"After repartition(16) : {upscaled.rdd.getNumPartitions()}")
    print("Exchange IS required here — repartition is the correct choice")

# ── Step 6: Column-based repartition for join pre-partitioning ───────────────
# Downstream join on LicenseType? Pre-partition so the join skips its Exchange.
col_reprt = active.repartition(4, "LicenseType")
print("\n=== repartition(4, 'LicenseType') — HashPartitioning Exchange ===")
col_reprt.explain(mode="formatted")
# Exchange HashPartitioning(LicenseType, 4): same key → same partition
# A subsequent join or groupBy on LicenseType can eliminate its own Exchange

---
## Topic 5 — CSV vs Parquet

### Business Scenario
| Dimension | Detail |
|-----------|--------|
| **Team** | TLC Data Platform — Reporting & ML Feature Pipeline |
| **Goal** | Read cab registry 100 times/day for dashboards and ML feature extraction |
| **Data Volume** | 500K rows × 14 columns simulated — makes format differences measurable locally |
| **Pain Point** | CSV: no compression, no embedded schema, no column stats, full row scan every read |
| **Business Impact** | Switching to Parquet reduces daily I/O from ~85 MB × 100 = 8.5 GB to ~800 MB (91% savings) |

### Format Comparison
```
Dimension              CSV                       Parquet
──────────────────────────────────────────────────────────────────────────
Orientation            Row-based                 Columnar (column chunks)
Compression            None / gzip per file      Snappy/ZSTD per column
Schema                 Inferred (extra scan)     Embedded in file footer
Predicate Pushdown     NO                        YES (row-group min/max stats)
Column Pruning I/O     NO (must read all bytes)  YES (skip column chunks)
Vectorized read        NO  (Batched: false)       YES (Batched: true, Arrow)
Splittable             YES (uncompressed)         YES
Human-readable         YES                       NO (binary)
Write speed            Fast                      Slightly slower
Read speed (filter)    Slow — full scan          Fast — pushdown + pruning
Schema evolution       Manual                    Native (nullable columns)
Typical compression    1× (raw text)             3–10× smaller
```

### Why Parquet Wins at Scale
```
500K rows × 14 columns:
  CSV    size: ~85 MB  (plain text, no compression)
  Parquet size: ~8 MB  (Snappy compressed, columnar)

Query: SELECT LicenseType, count(*) WHERE Active='YES' GROUP BY LicenseType
  CSV   I/O: 85 MB  (reads all 14 columns, all rows)
  Parquet I/O: ~1.2 MB  (reads 2 of 14 column chunks, pushes filter to reader)
  Speedup: ~70× on this query alone
```

In [ ]:
# ── Topic 5: Generate 500K-row synthetic fleet dataset ───────────────────────
# We need enough data to make CSV vs Parquet differences measurable on a laptop
import os, glob

csv_path     = os.path.join(SCRATCH, "t5_large_fleet_csv")
parquet_path = os.path.join(SCRATCH, "t5_large_fleet_parquet")

license_types = ["OWNER MUST DRIVE", "NAMED DRIVER", "CORPORATE"]

large_fleet = (
    spark.range(500_000)
         .withColumn("CabNumber",            F.concat(F.lit("T"), F.col("id").cast("string"), F.lit("C")))
         .withColumn("VehicleLicenseNumber",  F.concat(F.lit("LIC"), F.col("id").cast("string")))
         .withColumn("Name",                  F.concat(F.lit("OPERATOR "), F.col("id").cast("string")))
         .withColumn("LicenseType",           F.element_at(
             F.array([F.lit(x) for x in license_types]),
             (F.col("id") % 3 + 1).cast("int")))
         .withColumn("Active",                F.when(F.col("id") % 5 == 0, "NO").otherwise("YES"))
         .withColumn("PermitLicenseNumber",   F.when(F.col("id") % 3 == 0,
             F.concat(F.lit("PERM"), F.col("id").cast("string"))).otherwise(F.lit(None)))
         .withColumn("VehicleVinNumber",      F.concat(F.lit("VIN"), F.col("id").cast("string")))
         .withColumn("WheelchairAccessible",  F.when(F.col("id") % 10 == 0, F.lit("YES")).otherwise(F.lit(None)))
         .withColumn("VehicleYear",           (F.lit(2010) + (F.col("id") % 14)).cast("int"))
         .withColumn("VehicleType",           F.when(F.col("id") % 4 == 0, F.lit("Sedan"))
                                               .when(F.col("id") % 4 == 1, F.lit("SUV"))
                                               .otherwise(F.lit(None)))
         .withColumn("TelephoneNumber",       F.concat(F.lit("(718)555-"), F.lpad(F.col("id").cast("string"), 4, "0")))
         .withColumn("Website",               F.when(F.col("id") % 5 == 0,
             F.concat(F.lit("http://cab"), F.col("id").cast("string"), F.lit(".com"))).otherwise(F.lit(None)))
         .withColumn("Address",               F.concat(F.col("id").cast("string"), F.lit(" MAIN ST NEW YORK NY 10001")))
         .withColumn("LastDateUpdated",       F.lit("04/22/2020"))
         .drop("id")
)

# Write CSV (coalesce to 1 file for fair single-file size comparison)
large_fleet.coalesce(1).write.mode("overwrite").option("header", "true").csv(csv_path)
# Write Parquet
large_fleet.write.mode("overwrite").parquet(parquet_path)

csv_file    = glob.glob(csv_path + "/part-*.csv")[0]
parquet_files = glob.glob(parquet_path + "/part-*.parquet")
csv_mb      = os.path.getsize(csv_file) / 1024 / 1024
parquet_mb  = sum(os.path.getsize(f) for f in parquet_files) / 1024 / 1024

print(f"Dataset: 500,000 rows × 14 columns")
print(f"CSV     size: {csv_mb:.1f} MB")
print(f"Parquet size: {parquet_mb:.1f} MB")
print(f"Compression ratio: {csv_mb/max(parquet_mb, 0.001):.1f}×  (Snappy)")

### Problem Statement
The reporting team stores the full fleet as CSV and reads it 100 times per day.
Every read incurs: full schema inference (extra scan), reads all 14 columns even for 1-column queries,
no predicate pushdown, no vectorized (Arrow) read path.

Your job: benchmark CSV vs Parquet reads with a filter+aggregation query, run `explain()` for both,
and quantify the I/O savings.

In [ ]:
# ── BAD CODE: CSV without schema, no pushdown ────────────────────────────────
import os, glob
csv_path = os.path.join(SCRATCH, "t5_large_fleet_csv")
csv_file = glob.glob(csv_path + "/part-*.csv")[0]

# PROBLEM 1: No schema → Spark reads CSV twice (inference scan + query scan)
# PROBLEM 2: CSV cannot push predicates to the reader — full scan always
# PROBLEM 3: Batched: false — row-by-row processing, no Arrow vectorization

t0 = time.time()
bad_df = (
    spark.read
         .option("header", "true")   # no schema = inference scan triggered
         .csv(csv_file)              # reads entire file
         .filter(F.col("Active") == "YES")   # filter AFTER full deserialization
         .groupBy("LicenseType")
         .count()
)
bad_result = bad_df.count()
t_csv_bad = time.time() - t0

print(f"CSV (no schema, no pushdown): {t_csv_bad:.3f}s  groups={bad_result}")
print("\n=== BAD CSV plan — PushedFilters:[], Batched:false ===")
spark.read.option("header","true").csv(csv_file) \
     .filter(F.col("Active") == "YES") \
     .groupBy("LicenseType").count() \
     .explain(mode="formatted")

### Interview Questions — Topic 5: CSV vs Parquet

1. Explain the internal structure of a Parquet file: row groups, column chunks, pages, and the file footer. How does each level enable different query optimizations?
2. Why does Parquet support predicate pushdown while CSV does not? What specific metadata in the Parquet footer/row-group header enables row-group skipping?
3. Schema inference from CSV triggers an extra scan. Exactly when is this scan triggered, and what is the performance cost at 1 TB scale?
4. You have a 10 GB CSV file with 50 columns. Your query aggregates 2 columns. Compare the I/O cost of CSV vs Parquet for this specific query.
5. Parquet uses Snappy compression by default in Spark. When would you switch to ZSTD or GZIP, and what are the read vs write speed trade-offs?
6. What is Parquet **dictionary encoding** and when does it activate? Which column type benefits most from it?
7. You write a DataFrame to Parquet and then read it back with a different schema (one new nullable column added). How does Parquet handle this schema evolution without rewriting the file?
8. Compare `Batched: true` vs `Batched: false` in the physical plan. What underlying Spark/Arrow mechanism does vectorized reading use, and what is the typical CPU speedup?

### Expected Logical Plan Discussion

```
CSV plan (Optimized):
Aggregate [LicenseType], [LicenseType, count(1)]
+- Filter (isnotnull(Active) AND (Active = YES))
   +- Relation [...all 14 cols...] csv        <-- full row read, no stats available

Parquet plan (Optimized):
Aggregate [LicenseType], [LicenseType, count(1)]
+- Project [LicenseType]                      <-- ColumnPruning pushed down
   +- Relation [Active, LicenseType] parquet  <-- ReadSchema: 2 of 14 cols; PushedFilters set
```

**Key differences:**
- CSV: `Filter` node sits above `Relation` — filter runs in Spark after full byte deserialization
- Parquet: filter is absorbed into `Relation` as `PushedFilters` — the Parquet reader handles it
- Parquet: `ColumnPruning` narrows `Relation` to 2 columns; CSV `Relation` still has all 14
- CSV `Relation` type is `csv` — Catalyst knows this format has no pushdown capability
- Parquet `Relation` type is `parquet` — Catalyst applies both `PushDownPredicates` and `ColumnPruning`

### Expected Physical Plan Discussion

```
CSV physical plan:
*(2) HashAggregate(keys=[LicenseType], functions=[count(1)])
+- Exchange hashpartitioning(LicenseType, 8)
   +- *(1) HashAggregate(keys=[LicenseType], functions=[partial_count(1)])
      +- *(1) Filter (isnotnull(Active) AND (Active = YES))
         +- *(1) FileScan csv [Active, LicenseType, ...14 cols]
                  Batched: false                    <-- row-by-row, no Arrow
                  PushedFilters: []                 <-- EMPTY — no pushdown possible
                  ReadSchema: struct<...14 cols...> <-- all columns decoded

Parquet physical plan:
*(2) HashAggregate(keys=[LicenseType], functions=[count(1)])
+- Exchange hashpartitioning(LicenseType, 8)
   +- *(1) HashAggregate(keys=[LicenseType], functions=[partial_count(1)])
      +- *(1) FileScan parquet [Active, LicenseType]
               Batched: true                              <-- vectorized Arrow read
               PushedFilters: [IsNotNull(Active), EqualTo(Active,YES)]  <-- POPULATED
               ReadSchema: struct<Active:string, LicenseType:string>    <-- 2 cols only!
```

**Side-by-side metrics:**
| Metric | CSV | Parquet | Impact |
|--------|-----|---------|--------|
| `Batched` | false | true | Parquet uses Arrow columnar batches — ~10× faster CPU |
| `PushedFilters` | empty | populated | Parquet skips row groups inside the file reader |
| `ReadSchema` | 14 cols | 2 cols | 6× less I/O for this specific query |
| `Input Size` (UI) | ~85 MB | ~1.2 MB | 98.6% I/O reduction |

### Spark UI Investigation Guide — CSV vs Parquet

**SQL tab → FileScan node tooltip:**
- `Format: CSV` vs `Format: Parquet` — immediately identifies the source format
- `Batched: false` (CSV) vs `Batched: true` (Parquet) — vectorized reads are ~10× faster CPU utilization
- `PushedFilters: []` (CSV) vs `PushedFilters: [EqualTo(...)]` (Parquet)
- `ReadSchema` — for Parquet should contain only the selected columns

**Stages tab — Input Size:**
- CSV query: Input Size ≈ full file size, regardless of filter selectivity or column count
- Parquet query: Input Size ≈ (selected_columns / total_columns) × compressed_size × filter_selectivity

**Jobs tab — number of jobs:**
- CSV with schema inference: you may see an extra job for the inference scan before the main job
- Parquet: single job per query

**Key metrics to record in interviews:**
```
500K rows, filter Active=YES (80% selectivity), GROUP BY LicenseType:

CSV (no schema):  Input = 85 MB,  Duration = 3.8s, Batched=false, PushedFilters=[]
CSV (w/ schema):  Input = 85 MB,  Duration = 2.1s, Batched=false, PushedFilters=[]
Parquet:          Input = 1.2 MB, Duration = 0.3s, Batched=true,  PushedFilters=[IsNotNull, EqualTo]
Speedup CSV→Parquet: ~7-13× on 500K rows; grows to 50-100× at 50M+ rows
```

In [ ]:
# ── OPTIMIZATION EXERCISE ────────────────────────────────────────────────────
import os, glob
csv_path     = os.path.join(SCRATCH, "t5_large_fleet_csv")
parquet_path = os.path.join(SCRATCH, "t5_large_fleet_parquet")
csv_file     = glob.glob(csv_path + "/part-*.csv")[0]

fleet_schema = StructType([
    StructField("CabNumber",            StringType(),  True),
    StructField("VehicleLicenseNumber", StringType(),  True),
    StructField("Name",                 StringType(),  True),
    StructField("LicenseType",          StringType(),  True),
    StructField("Active",               StringType(),  True),
    StructField("PermitLicenseNumber",  StringType(),  True),
    StructField("VehicleVinNumber",     StringType(),  True),
    StructField("WheelchairAccessible", StringType(),  True),
    StructField("VehicleYear",          IntegerType(), True),
    StructField("VehicleType",          StringType(),  True),
    StructField("TelephoneNumber",      StringType(),  True),
    StructField("Website",              StringType(),  True),
    StructField("Address",              StringType(),  True),
    StructField("LastDateUpdated",      StringType(),  True),
])

# TODO 1: Read CSV with explicit schema (avoid inference — saves one full scan)
# TODO 2: Read Parquet with filter + column selection
# TODO 3: Call .explain(mode='formatted') on both — compare:
#         Batched, PushedFilters, ReadSchema, number of columns
# TODO 4: Time both for: filter Active=YES, groupBy LicenseType, count
# TODO 5: Compute speedup ratio

# YOUR CODE HERE

In [ ]:
# ── PERFORMANCE BENCHMARKING — CSV vs Parquet ────────────────────────────────
import os, glob
csv_path     = os.path.join(SCRATCH, "t5_large_fleet_csv")
parquet_path = os.path.join(SCRATCH, "t5_large_fleet_parquet")
csv_file     = glob.glob(csv_path + "/part-*.csv")[0]

fleet_schema = StructType([
    StructField("CabNumber",            StringType(),  True),
    StructField("VehicleLicenseNumber", StringType(),  True),
    StructField("Name",                 StringType(),  True),
    StructField("LicenseType",          StringType(),  True),
    StructField("Active",               StringType(),  True),
    StructField("PermitLicenseNumber",  StringType(),  True),
    StructField("VehicleVinNumber",     StringType(),  True),
    StructField("WheelchairAccessible", StringType(),  True),
    StructField("VehicleYear",          IntegerType(), True),
    StructField("VehicleType",          StringType(),  True),
    StructField("TelephoneNumber",      StringType(),  True),
    StructField("Website",              StringType(),  True),
    StructField("Address",              StringType(),  True),
    StructField("LastDateUpdated",      StringType(),  True),
])

RUNS = 3

def bench(label, fn):
    times = []
    for _ in range(RUNS):
        t0 = time.time()
        fn()
        times.append(time.time() - t0)
    avg = sum(times) / len(times)
    print(f"[{label:<40}] avg={avg:.3f}s  runs={[f'{t:.3f}' for t in times]}")
    return avg

t_csv_infer = bench(
    "CSV — no schema (inference + scan)",
    lambda: spark.read.option("header","true").csv(csv_file)
                 .filter(F.col("Active")=="YES")
                 .groupBy("LicenseType").count().collect()
)

t_csv_schema = bench(
    "CSV — explicit schema (scan only)",
    lambda: spark.read.csv(csv_file, header=True, schema=fleet_schema)
                 .filter(F.col("Active")=="YES")
                 .groupBy("LicenseType").count().collect()
)

t_parquet = bench(
    "Parquet — pushdown + pruning",
    lambda: spark.read.parquet(parquet_path)
                 .filter(F.col("Active")=="YES")
                 .groupBy("LicenseType").count().collect()
)

print(f"\n{'Method':<42} {'Speedup vs CSV-infer':>22}")
print("-" * 66)
print(f"{'CSV — no schema':<42} {'1.0×':>22}")
print(f"{'CSV — explicit schema':<42} {t_csv_infer/max(t_csv_schema,0.001):>21.1f}×")
print(f"{'Parquet — pushdown + pruning':<42} {t_csv_infer/max(t_parquet,0.001):>21.1f}×")

print("\n=== Parquet plan — PushedFilters and ReadSchema ===")
spark.read.parquet(parquet_path) \
     .filter(F.col("Active") == "YES") \
     .groupBy("LicenseType").count() \
     .explain(mode="formatted")

### Production Best Practices — CSV vs Parquet

- **Never store production datasets as CSV** if they will be read more than once. Convert to Parquet or Delta Lake at the ETL landing zone — the one-time write cost is recovered on the first few reads.
- **Always specify the schema when reading CSV.** Inference adds a full extra scan. At 1 TB scale this doubles your CSV I/O before the query even starts. Store schemas in code or a schema registry.
- **Use Parquet with Snappy compression as your default.** Snappy offers a good balance: 3-5× compression ratio with fast decompression (CPU-efficient, no blocking). Use ZSTD for maximum compression on cold archival storage.
- **Set `spark.sql.parquet.compression.codec=snappy`** (default) for hot data, `zstd` for archival. Avoid `gzip` for Parquet — while it compresses better, it is slower to decompress and creates non-splittable pages in some implementations.
- **Partition Parquet tables by date or a low-cardinality filter column.** This layers partition pruning (Topic 6) on top of column pruning and predicate pushdown for maximum I/O reduction.
- **Set `spark.sql.parquet.mergeSchema=false`** (default). Schema merging scans all Parquet footers on every read — catastrophically expensive on tables with thousands of files.
- **For small lookup tables (< 100 MB)**, CSV is acceptable — conversion overhead is not worth it. But always specify the schema explicitly to avoid the inference scan.

### Common Interview Follow-up Questions — CSV vs Parquet

1. Delta Lake is built on Parquet. What does Delta add on top of Parquet that raw Parquet alone cannot provide (ACID transactions, time travel, schema enforcement, DML operations)?
2. You need to share data with a non-Spark system that cannot read Parquet. What are your options for balancing readability and performance — ORC, Avro, JSON Lines, Parquet with a converter tool?
3. Parquet stores column statistics (min, max, null count) at the row-group level. What is the default row group size, how does `spark.sql.parquet.block.size` control it, and what happens to pushdown effectiveness if rows groups are too large?
4. You have a Parquet table with 1,000 small files (1 MB each). Total data is 1 GB but reads are slow. What are the two distinct performance problems and how do you fix each independently?
5. Compare Parquet and ORC as columnar formats. When would you choose ORC over Parquet in a Spark-on-Hive workload, and vice versa?

In [ ]:
# ============ SOLUTION (scroll down) ============

# ── SOLUTION: CSV vs Parquet — Complete Comparison ───────────────────────────
import os, glob
csv_path     = os.path.join(SCRATCH, "t5_large_fleet_csv")
parquet_path = os.path.join(SCRATCH, "t5_large_fleet_parquet")
csv_file     = glob.glob(csv_path + "/part-*.csv")[0]

fleet_schema = StructType([
    StructField("CabNumber",            StringType(),  True),
    StructField("VehicleLicenseNumber", StringType(),  True),
    StructField("Name",                 StringType(),  True),
    StructField("LicenseType",          StringType(),  True),
    StructField("Active",               StringType(),  True),
    StructField("PermitLicenseNumber",  StringType(),  True),
    StructField("VehicleVinNumber",     StringType(),  True),
    StructField("WheelchairAccessible", StringType(),  True),
    StructField("VehicleYear",          IntegerType(), True),
    StructField("VehicleType",          StringType(),  True),
    StructField("TelephoneNumber",      StringType(),  True),
    StructField("Website",              StringType(),  True),
    StructField("Address",              StringType(),  True),
    StructField("LastDateUpdated",      StringType(),  True),
])

# ── Approach A: Best-practice CSV — explicit schema, no inference ────────────
t0 = time.time()
csv_result = (
    spark.read
         .csv(csv_file, header=True, schema=fleet_schema)  # explicit schema = 1 scan, not 2
         .filter(F.col("Active") == "YES")                 # DataFilter post-deserialization
         .select("LicenseType", "VehicleYear")             # ColumnPruning (no I/O benefit for CSV)
         .groupBy("LicenseType")
         .agg(F.count("*").alias("cabs"), F.avg("VehicleYear").alias("avg_year"))
         .collect()
)
t_csv = time.time() - t0

# ── Approach B: Parquet — vectorized read + pushdown + column pruning ────────
t0 = time.time()
parquet_result = (
    spark.read
         .parquet(parquet_path)                            # Batched:true — Arrow columnar
         .filter(F.col("Active") == "YES")                 # PushedFilters: inside Parquet reader
         .select("LicenseType", "VehicleYear")             # ReadSchema: 3 of 14 cols
         .groupBy("LicenseType")
         .agg(F.count("*").alias("cabs"), F.avg("VehicleYear").alias("avg_year"))
         .collect()
)
t_parquet = time.time() - t0

print(f"{'Method':<38} {'Time (s)':>10} {'Speedup':>10}")
print("-" * 62)
print(f"{'CSV  + explicit schema':<38} {t_csv:>10.3f} {'1.0×':>10}")
print(f"{'Parquet + pushdown + pruning':<38} {t_parquet:>10.3f} {t_csv/max(t_parquet,0.001):>9.1f}×")

# Verify correctness
csv_map     = {r['LicenseType']: r['cabs'] for r in csv_result}
parquet_map = {r['LicenseType']: r['cabs'] for r in parquet_result}
print(f"\nResults match: {csv_map == parquet_map}")
for lt in sorted(parquet_map):
    print(f"  {lt:<25}: {parquet_map[lt]:,} cabs")

# ── Explain comparison ───────────────────────────────────────────────────────
print("\n=== CSV plan — Batched:false, PushedFilters:[] ===")
spark.read.csv(csv_file, header=True, schema=fleet_schema) \
     .filter(F.col("Active") == "YES") \
     .groupBy("LicenseType").count() \
     .explain(mode="formatted")

print("\n=== Parquet plan — Batched:true, PushedFilters populated ===")
spark.read.parquet(parquet_path) \
     .filter(F.col("Active") == "YES") \
     .groupBy("LicenseType").count() \
     .explain(mode="formatted")

---
## Topic 6 — Partition Pruning

### Business Scenario
| Dimension | Detail |
|-----------|--------|
| **Team** | TLC Data Warehouse — Analytical Queries |
| **Goal** | Efficiently query cab data by `Active` status and `VehicleYear` without full table scan |
| **Data Volume** | 500K rows written to Parquet partitioned by `Active` and `VehicleYear` |
| **Pain Point** | Without Hive-style partitioning, every query scans all files regardless of year filter |
| **Impact** | Partition pruning on `VehicleYear=2020` skips 27 of 28 partition directories (96% I/O savings) |

### How Partition Pruning Works
When you write Parquet with `.partitionBy('col')`, Spark creates a directory hierarchy:
```
output/
  Active=NO/
    VehicleYear=2010/  part-00000.parquet
    VehicleYear=2011/  part-00000.parquet
    ...
  Active=YES/
    VehicleYear=2010/  part-00000.parquet
    VehicleYear=2011/  part-00000.parquet
    ...
```

When you query `WHERE Active='YES' AND VehicleYear=2020`, Spark's **file listing step**
skips all other directories before a single byte is read. This is `PartitionFilters`.

### PartitionFilters vs PushedFilters
```
PartitionFilters:  Evaluated at FILE LISTING time — entire directories are skipped.
                   Zero I/O for skipped partitions (not even file headers opened).
                   Only works on columns used in .partitionBy().

PushedFilters:     Evaluated inside the Parquet reader — row groups within a file are skipped.
                   Reads file footer for statistics, then skips non-matching row groups.
                   Works on any column with column statistics in the Parquet footer.

PartitionFilters >> PushedFilters in terms of data eliminated per unit of work.
```

### Partitioning Anti-Pattern
```
BAD:  .partitionBy('CabNumber')  -- 8,970 unique values → 8,970 tiny directories
      Each directory has 1 row. Useless for query pruning. Catastrophic metadata overhead.

GOOD: .partitionBy('Active', 'VehicleYear')  -- 2 × 14 = 28 directories
      Each directory has ~18,000 rows. Selective and well-sized.

Rule: cardinality × avg_rows_per_partition >> 1 Parquet row group (128 MB default)
```

In [ ]:
# ── Topic 6: Write partitioned and unpartitioned Parquet for comparison ───────
import os, glob

fleet_path           = os.path.join(SCRATCH, "t3_large_fleet")
partitioned_path     = os.path.join(SCRATCH, "t6_fleet_partitioned")
unpartitioned_path   = os.path.join(SCRATCH, "t6_fleet_unpartitioned")
bad_partition_path   = os.path.join(SCRATCH, "t6_fleet_bad_partition")

fleet = spark.read.parquet(fleet_path)

# GOOD: partition by low-cardinality columns (2 × 14 = 28 partition directories)
fleet.write.mode("overwrite") \
     .partitionBy("Active", "VehicleYear") \
     .parquet(partitioned_path)

# BAD: partition by a higher-cardinality column that queries never filter on alone
# Using 'Borough' (5 values) to simulate a wrong partition key choice —
# queries on VehicleYear get NO pruning benefit
fleet.write.mode("overwrite") \
     .partitionBy("Borough") \
     .parquet(bad_partition_path)

# UNPARTITIONED: baseline for comparison
fleet.write.mode("overwrite").parquet(unpartitioned_path)

# Inspect directory structure
top_dirs    = glob.glob(partitioned_path + "/*")
yes_subdirs = glob.glob(partitioned_path + "/Active=YES/*")
no_subdirs  = glob.glob(partitioned_path + "/Active=NO/*")
bad_dirs    = glob.glob(bad_partition_path + "/*")

print(f"GOOD partition — top-level dirs (Active=YES/NO): {len(top_dirs)}")
print(f"  Active=YES year dirs: {len(yes_subdirs)}")
print(f"  Active=NO  year dirs: {len(no_subdirs)}")
print(f"  Total partition dirs: {len(yes_subdirs)+len(no_subdirs)}")
print(f"\nBAD  partition — Borough dirs: {len(bad_dirs)}")
print(f"  Sample: {sorted(bad_dirs)[:3]}")
print(f"\nSample GOOD partition paths (Active=YES, first 3 years):")
for d in sorted(yes_subdirs)[:3]:
    files = glob.glob(d + "/*.parquet")
    sz_kb = sum(os.path.getsize(f) for f in files) / 1024
    print(f"  .../{d.split(os.sep)[-2]}/{d.split(os.sep)[-1]}/ — {len(files)} file(s), {sz_kb:.1f} KB")

### Problem Statement
The analytics team queries cab data filtered by `VehicleYear` daily.
The bad implementation reads an unpartitioned Parquet table — full scan every time.
A junior engineer tried to fix performance by partitioning by `Borough` — still a full
scan for `VehicleYear` queries because `Borough` is not the filter column.

Your job: write correctly partitioned Parquet, show `PartitionFilters` in the physical plan,
and quantify the I/O reduction.

In [ ]:
# ── BAD CODE: unpartitioned scan + wrong partition key ───────────────────────
import os
unpartitioned_path = os.path.join(SCRATCH, "t6_fleet_unpartitioned")
bad_partition_path = os.path.join(SCRATCH, "t6_fleet_bad_partition")

# BAD 1: No partitioning — VehicleYear filter forces full table scan
t0 = time.time()
n_unpart = (
    spark.read.parquet(unpartitioned_path)
         .filter((F.col("Active") == "YES") & (F.col("VehicleYear") == 2020))
         .count()
)
t_unpart = time.time() - t0
print(f"Unpartitioned : {n_unpart:,} rows  {t_unpart:.3f}s")

print("\n=== Unpartitioned plan — PartitionFilters: [] ===")
spark.read.parquet(unpartitioned_path) \
     .filter((F.col("Active") == "YES") & (F.col("VehicleYear") == 2020)) \
     .explain(mode="formatted")

# BAD 2: Wrong partition key (Borough) — VehicleYear query still scans all 5 Borough dirs
t0 = time.time()
n_bad_part = (
    spark.read.parquet(bad_partition_path)
         .filter((F.col("Active") == "YES") & (F.col("VehicleYear") == 2020))
         .count()
)
t_bad_part = time.time() - t0
print(f"Bad partition (Borough): {n_bad_part:,} rows  {t_bad_part:.3f}s")

print("\n=== Bad partition plan — PartitionFilters empty for VehicleYear query ===")
spark.read.parquet(bad_partition_path) \
     .filter((F.col("Active") == "YES") & (F.col("VehicleYear") == 2020)) \
     .explain(mode="formatted")

### Interview Questions — Topic 6: Partition Pruning

1. What is Hive-style (directory) partitioning in Parquet, and how does Spark discover partition values without reading any file data?
2. Explain the difference between `PartitionFilters` and `PushedFilters` in a Spark physical plan. At exactly what stage of execution does each one eliminate data?
3. You have a table partitioned by `date` (3,650 directories for 10 years). A query filters `date BETWEEN '2023-01-01' AND '2023-03-31'`. How many partition directories does Spark open, and how does it determine this before reading any file contents?
4. A table is partitioned by `country` (200 values) then `date` (3,650 values) — 730,000 total directories. Queries always filter on both. What is the problem with this partitioning strategy and how do you restructure it?
5. What is `spark.sql.sources.partitionOverwriteMode`? Explain `static` vs `dynamic` and when each mode is dangerous.
6. You filter on a partition column using a UDF: `df.filter(my_udf(col('VehicleYear')) == 2020)`. Does partition pruning still work? Explain why or why not at the execution level.
7. `spark.sql.optimizer.metadataOnly` is enabled. A query runs `SELECT DISTINCT Active FROM partitioned_table`. What optimization occurs, and when does it return incorrect results?
8. Your table is partitioned by `date`. A query filters by `date` AND also has `LicenseType='CORPORATE'`. Describe the full execution — which skipping happens at which layer (file listing vs Parquet reader vs Spark filter)?

### Expected Logical Plan Discussion

```
Unpartitioned Parquet (Optimized):
Filter (isnotnull(Active) AND (Active = YES) AND isnotnull(VehicleYear) AND (VehicleYear = 2020))
+- Relation [...] parquet        <-- ALL files scanned; filter runs post-deserialization

Partitioned by Active + VehicleYear (Optimized):
Relation [CabNumber, LicenseType, ...non-partition cols...] parquet
  <-- The partition filter was absorbed at FILE LISTING time.
  <-- No separate Filter node needed for partition columns!
  <-- Spark listed only Active=YES/VehicleYear=2020/ directory.
```

**Key insight:** When the entire filter is on partition columns, Catalyst can eliminate
the `Filter` node from the logical plan entirely. The Spark file lister handles it:
1. Spark calls `listLeafFiles()` on the root path
2. Path pattern matching on directory names: `Active=YES/VehicleYear=2020`
3. Only that directory's files are added to the task list
4. All other directories are never opened, stat'd, or read — zero I/O

**Mixed filter (partition + non-partition column):**
```
Filter (LicenseType = CORPORATE)                  <-- PushedFilters or DataFilters
+- Relation [Active=YES, VehicleYear=2020] parquet <-- PartitionFilters already applied
```
The partition filter eliminates directories (coarse skip); the non-partition filter
eliminates row groups within the remaining files (fine-grained skip).

### Expected Physical Plan Discussion

```
Unpartitioned:
== Physical Plan ==
*(1) FileScan parquet [CabNumber, LicenseType, Active, VehicleYear, ...]
     PartitionFilters: []                    <-- NO pruning — all files read
     PushedFilters: [IsNotNull(Active), EqualTo(Active,YES),
                     IsNotNull(VehicleYear), EqualTo(VehicleYear,2020)]
     ReadSchema: struct<...>
     number of files: 8                     <-- ALL Parquet files scanned

Partitioned by Active + VehicleYear:
== Physical Plan ==
*(1) FileScan parquet [CabNumber, LicenseType, Borough, Trips30Days,
                       Active, VehicleYear]
     PartitionFilters: [isnotnull(Active), (Active = YES),
                        isnotnull(VehicleYear), (VehicleYear = 2020)]  <-- POPULATED!
     PushedFilters: []                      <-- nothing left to push
     ReadSchema: struct<CabNumber:string, LicenseType:string, ...>
     number of files: 1                     <-- only 1 of 28 partition dirs!
```

**Physical plan node metrics to compare:**
| Metric | Unpartitioned | Partitioned | Impact |
|--------|---------------|-------------|--------|
| `PartitionFilters` | empty | populated | Directories skipped at listing time |
| `number of files` | all files | 1 file | 27 of 28 directories skipped |
| `Input Size` (UI) | ~18 MB | ~0.65 MB | 96% I/O reduction |
| `PushedFilters` | populated | empty | Partition filter subsumed the row-group filter |

### Spark UI Investigation Guide — Partition Pruning

**SQL tab → FileScan node tooltip:**
- `PartitionFilters` — if populated, partition pruning is active
- `number of files read` — should be a small fraction of total files for selective queries
- `size of files read` — compare against total table size; large gap means pruning is working

**Stages tab:**
- Unpartitioned scan: `Input Size` ≈ full table size
- Partitioned scan with selective filter: `Input Size` ≈ (matching_partitions / total_partitions) × table_size

**Environment tab → Spark Properties:**
- `spark.sql.sources.partitionOverwriteMode` — should be `dynamic` for incremental append pipelines
- `spark.sql.optimizer.metadataOnly` — enabled by default; can return incorrect `DISTINCT` counts on partitioned tables with deleted rows

**Key metrics to record in interviews:**
```
Query: Active=YES AND VehicleYear=2020  (1 of 28 partition directories)

Unpartitioned : files=8,  Input=18 MB,  Duration=1.8s, PartitionFilters=[]
Partitioned   : files=1,  Input=0.65 MB, Duration=0.3s, PartitionFilters=[EqualTo...]
Pruning saves : ~97% I/O, ~6× faster  (grows to 100× at TB scale)
```

**Diagnosing partition discovery overhead:**
If a table has thousands of partitions (e.g., daily date partitions over 10 years = 3,650 dirs),
the filesystem `listStatus` call itself becomes a bottleneck. In that case, register the table
in a metastore (Hive, AWS Glue, Unity Catalog) so Spark reads partition paths from the catalog
instead of listing the filesystem.

In [ ]:
# ── OPTIMIZATION EXERCISE ────────────────────────────────────────────────────
import os
fleet_path = os.path.join(SCRATCH, "t3_large_fleet")

fleet = spark.read.parquet(fleet_path)

# TODO 1: Write fleet to Parquet partitioned by Active then VehicleYear
#         my_partitioned = os.path.join(SCRATCH, "t6_exercise_partitioned")
# TODO 2: Query: filter Active=YES AND VehicleYear=2022
# TODO 3: Call .explain(mode='formatted') — verify PartitionFilters is populated
# TODO 4: Verify 'number of files' in the plan is 1 (not all 28)
# TODO 5: Time the partitioned vs unpartitioned query and compute speedup
# TODO 6: Demonstrate UDF on partition column blocks pruning:
#         df.filter(udf_fn(col('VehicleYear')) == 2022) — show PartitionFilters becomes empty

# YOUR CODE HERE

In [ ]:
# ── PERFORMANCE BENCHMARKING — Partition Pruning ─────────────────────────────
import os
partitioned_path   = os.path.join(SCRATCH, "t6_fleet_partitioned")
unpartitioned_path = os.path.join(SCRATCH, "t6_fleet_unpartitioned")

RUNS = 3

def time_query(label, df_fn, runs=RUNS):
    times = []
    for _ in range(runs):
        t0 = time.time()
        df_fn().count()
        times.append(time.time() - t0)
    avg = sum(times) / len(times)
    print(f"[{label:<50}] avg={avg:.3f}s")
    return avg

# Query 1: single year, single Active → maximum pruning (1/28 dirs)
t1_u = time_query(
    "Unpartitioned: Active=YES, VehicleYear=2020",
    lambda: spark.read.parquet(unpartitioned_path)
                 .filter((F.col("Active")=="YES") & (F.col("VehicleYear")==2020))
)
t1_p = time_query(
    "Partitioned:   Active=YES, VehicleYear=2020",
    lambda: spark.read.parquet(partitioned_path)
                 .filter((F.col("Active")=="YES") & (F.col("VehicleYear")==2020))
)

# Query 2: year range (3 years) — 6 of 28 dirs pruned
t2_u = time_query(
    "Unpartitioned: VehicleYear IN (2018,2019,2020)",
    lambda: spark.read.parquet(unpartitioned_path)
                 .filter(F.col("VehicleYear").isin(2018, 2019, 2020))
)
t2_p = time_query(
    "Partitioned:   VehicleYear IN (2018,2019,2020)",
    lambda: spark.read.parquet(partitioned_path)
                 .filter(F.col("VehicleYear").isin(2018, 2019, 2020))
)

print(f"\nQuery 1 speedup (1/28 dirs  pruned): {t1_u/max(t1_p, 0.001):.1f}×")
print(f"Query 2 speedup (6/28 dirs  pruned): {t2_u/max(t2_p, 0.001):.1f}×")

print("\n=== Partitioned plan — PartitionFilters populated ===")
spark.read.parquet(partitioned_path) \
     .filter((F.col("Active")=="YES") & (F.col("VehicleYear")==2020)) \
     .explain(mode="formatted")

### Production Best Practices — Partition Pruning

- **Choose partition columns with low-to-medium cardinality.** The practical range is 10–10,000 unique partition values. Too few (2 values) gives coarse pruning; too many (millions) creates metadata explosion and slow directory listing.
- **Partition by the most common query filter column first.** For time-series data this is almost always `date` or `year`. Analyze real query patterns before committing to a partition scheme — it is expensive to rewrite a large table.
- **Never partition by a high-cardinality column** like user ID, cab number, or VIN. Each unique value becomes its own directory with effectively one row — the worst possible outcome for both pruning and file size.
- **Use `partitionOverwriteMode=dynamic`** for incremental writes: `spark.conf.set('spark.sql.sources.partitionOverwriteMode', 'dynamic')`. Static mode (default) deletes ALL partitions before writing, even ones not in the current job — catastrophic for production incremental loads.
- **Register partitioned tables in a metastore** (Hive, AWS Glue, Unity Catalog). Without a metastore, Spark must list the filesystem on every read. For a table with 10,000+ partitions this listing takes seconds and dominates query latency.
- **Combine partitioning with `ZORDER BY`** (Delta Lake) or Parquet Bloom filters on non-partition columns. Partition pruning handles the coarse directory skip; ZORDER/Bloom handles the fine-grained skip within each partition file.
- **Avoid UDFs on partition columns in filter expressions.** A UDF on a partition column makes the filter opaque to Catalyst — partition pruning is disabled and all directories are scanned.

### Common Interview Follow-up Questions — Partition Pruning

1. Delta Lake's transaction log replaces Hive-style directory listing for partition discovery. How does this change the partition pruning behavior, and what is the performance implication for tables with millions of partitions?
2. You have a table partitioned by `date`. A query filters `YEAR(date) = 2023` using a Spark SQL function — not a literal equality on `date`. Does partition pruning fire? Why?
3. Explain `spark.sql.sources.partitionOverwriteMode=dynamic` in detail. What does "dynamic" mean for an `INSERT OVERWRITE` that touches only 3 of 365 daily partitions — which partitions are deleted?
4. A table partitioned by `country` has a subquery filter: `WHERE country IN (SELECT country FROM active_countries)`. Does partition pruning work with a subquery predicate? What optimization technique enables it?
5. You run `MSCK REPAIR TABLE` on a Hive table. What does this command do to the metastore, and why does it matter for partition pruning on a table whose partitions were added directly to the filesystem?

In [ ]:
# ============ SOLUTION (scroll down) ============

# ── SOLUTION: Partition Pruning — Complete Demonstration ─────────────────────
import os, glob
from pyspark.sql.functions import udf

partitioned_path   = os.path.join(SCRATCH, "t6_fleet_partitioned")
unpartitioned_path = os.path.join(SCRATCH, "t6_fleet_unpartitioned")
bad_partition_path = os.path.join(SCRATCH, "t6_fleet_bad_partition")

# ── Step 1: Confirm partition structure ──────────────────────────────────────
yes_dirs = glob.glob(partitioned_path + "/Active=YES/*")
no_dirs  = glob.glob(partitioned_path + "/Active=NO/*")
total    = len(yes_dirs) + len(no_dirs)
print(f"Partition directories: Active=YES → {len(yes_dirs)}, Active=NO → {len(no_dirs)}, total = {total}")

# ── Step 2: Correct query — PartitionFilters in action ──────────────────────
print("\n=== Partitioned plan: Active=YES, VehicleYear=2020 ===")
print("Look for 'PartitionFilters' — it should be populated, 'number of files' should be 1")
spark.read.parquet(partitioned_path) \
     .filter((F.col("Active") == "YES") & (F.col("VehicleYear") == 2020)) \
     .explain(mode="formatted")

# ── Step 3: Benchmark unpartitioned vs partitioned ───────────────────────────
t0 = time.time()
n_u = spark.read.parquet(unpartitioned_path) \
           .filter((F.col("Active")=="YES") & (F.col("VehicleYear")==2020)) \
           .count()
t_u = time.time() - t0

t0 = time.time()
n_p = spark.read.parquet(partitioned_path) \
           .filter((F.col("Active")=="YES") & (F.col("VehicleYear")==2020)) \
           .count()
t_p = time.time() - t0

print(f"\nUnpartitioned: {n_u:,} rows in {t_u:.3f}s  (scanned all {total} partition dirs)")
print(f"Partitioned  : {n_p:,} rows in {t_p:.3f}s  (scanned 1 of {total} partition dirs)")
print(f"Speedup      : {t_u/max(t_p,0.001):.1f}×  (scales to 100× at TB scale)")

# ── Step 4: Anti-pattern — UDF on partition column blocks pruning ─────────────
@udf(returnType=BooleanType())
def is_2020(year):
    """This UDF makes the VehicleYear filter opaque to the Spark planner."""
    return year == 2020

print("\n=== UDF on partition column — PartitionFilters BLOCKED ===")
spark.read.parquet(partitioned_path) \
     .filter((F.col("Active") == "YES") & is_2020(F.col("VehicleYear"))) \
     .explain(mode="formatted")
# PartitionFilters will be empty for VehicleYear — Catalyst cannot evaluate
# a UDF at file listing time. All 28 partition directories are listed and scanned.

# ── Step 5: Anti-pattern — wrong partition key (Borough) ─────────────────────
print("\n=== Bad partition key (Borough) — VehicleYear query gets NO pruning ===")
spark.read.parquet(bad_partition_path) \
     .filter((F.col("Active") == "YES") & (F.col("VehicleYear") == 2020)) \
     .explain(mode="formatted")
# PartitionFilters: [] for VehicleYear — it is not the partition key
# All 5 Borough directories are listed and all files are scanned

# ── Step 6: Mixed filter — partition + non-partition column ──────────────────
print("\n=== Mixed filter: partition col (VehicleYear) + non-partition col (LicenseType) ===")
spark.read.parquet(partitioned_path) \
     .filter(
         (F.col("Active") == "YES") &
         (F.col("VehicleYear") == 2020) &
         (F.col("LicenseType") == "CORPORATE")   # non-partition → PushedFilters
     ) \
     .explain(mode="formatted")
# PartitionFilters: [Active=YES, VehicleYear=2020]  → skips 27/28 dirs
# PushedFilters:    [EqualTo(LicenseType,CORPORATE)] → skips row groups within the 1 remaining file
# This is the two-layer skip: coarse (directories) + fine (row groups)

print("\nKEY INSIGHT: Partition pruning is the most powerful I/O optimization in Spark.")
print("It eliminates entire directory trees BEFORE any file is opened.")
print("Always partition by the columns your most frequent WHERE clauses filter on.")